In [ ]:
#block_1
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import GridSearchCV
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report

print("Loading data...")
train_df = pd.read_csv("train_dataset.csv").dropna(subset=['posts', 'labels'])
test_df = pd.read_csv("test_dataset.csv").dropna(subset=['posts', 'labels'])
lexicon_df = pd.read_excel("/content/Bangla_Depression_Lexicon_latest.xlsx").dropna(
    subset=['term_surface', 'category_name']
)

X_train = train_df['posts'].astype(str).values
y_train = train_df['labels'].astype(int).values
X_test = test_df['posts'].astype(str).values
y_test = test_df['labels'].astype(int).values

categories = sorted(lexicon_df['category_name'].unique())
cat_phrases = {
    cat: lexicon_df[lexicon_df['category_name'] == cat]['term_surface'].astype(str).tolist()
    for cat in categories
}

# Optional: severity-weighted lookup, only used if the column exists in your lexicon file
has_severity_weight = 'severity_association' in lexicon_df.columns
if has_severity_weight:
    cat_weights = {
        cat: dict(zip(
            lexicon_df[lexicon_df['category_name'] == cat]['term_surface'].astype(str),
            lexicon_df[lexicon_df['category_name'] == cat]['severity_association'].astype(float)
        ))
        for cat in categories
    }

PUNCT_RE = re.compile(r'[।\.,\?!;:"\'\(\)\[\]\{\}]')


def clean_text(text):
    # Replace with a space, not '', so punctuation doesn't fuse adjacent words
    return " " + PUNCT_RE.sub(' ', text).strip() + " "


class LexiconFeatureExtractor(BaseEstimator, TransformerMixin):
    """
    Per category, per document, produces:
      - raw match count
      - proportion of total lexicon matches (distribution across categories)
      - length-normalized density (matches / word count)
      - severity-weighted score (if the lexicon has a severity_association column)
    Scaler is fit once on train data and reused at transform time — never
    refit on whatever data happens to be passed in (this was the bug in v1).
    """

    def fit(self, X, y=None):
        raw = self._raw_features(X)
        self.scaler_ = MinMaxScaler().fit(raw)
        return self

    def transform(self, X):
        raw = self._raw_features(X)
        return self.scaler_.transform(raw)

    def _raw_features(self, X):
        features = []
        for text in X:
            text_clean = clean_text(text)
            word_count = max(len(text.split()), 1)

            cat_counts = []
            cat_weighted = []
            for cat in categories:
                phrases = cat_phrases[cat]
                count = sum(1 for phrase in phrases if f" {phrase} " in text_clean)
                cat_counts.append(count)

                if has_severity_weight:
                    weighted = sum(
                        cat_weights[cat].get(phrase, 0.0)
                        for phrase in phrases
                        if f" {phrase} " in text_clean
                    )
                    cat_weighted.append(weighted)

            total = sum(cat_counts)

            row = []
            row += cat_counts                                                 # raw counts
            row += [total]                                                    # total matches
            row += [(c / total) if total > 0 else 0.0 for c in cat_counts]    # proportion of total
            row += [c / word_count for c in cat_counts]                       # length-normalized density
            if has_severity_weight:
                row += cat_weighted                                           # severity-weighted score

            features.append(row)
        return np.array(features)


# Precompute lexicon features once, outside the grid search, since none of the
# grid parameters touch this extractor — avoids recomputing it on every one
# of the 36 param combos x 5 folds.
print("Precomputing lexicon features...")
lex_extractor = LexiconFeatureExtractor().fit(X_train)
X_train_lex = lex_extractor.transform(X_train)
X_test_lex = lex_extractor.transform(X_test)

# Grid search only over the TF-IDF + classifier stage
tfidf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(class_weight='balanced', random_state=42, max_iter=2000))
])

param_grid = {
    'tfidf__max_features': [3000, 5000, 10000],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df': [2, 5],
    'clf__C': [0.1, 1.0, 10.0]
}

print("Starting Grid Search on TF-IDF + classifier (lexicon features held fixed)...")
grid_search = GridSearchCV(
    tfidf_pipeline, param_grid, cv=5, scoring='f1_macro', n_jobs=-1, verbose=1
)

# Grid search tunes TF-IDF+clf on text alone; refit final model on
# TF-IDF(best params) + lexicon features concatenated together.
grid_search.fit(X_train, y_train)
print(f"\nBest CV Macro-F1 (TF-IDF only): {grid_search.best_score_:.4f}")
print(f"Winning TF-IDF/clf params: {grid_search.best_params_}")

best_tfidf_params = {
    k.replace('tfidf__', ''): v for k, v in grid_search.best_params_.items() if k.startswith('tfidf__')
}
best_C = grid_search.best_params_['clf__C']

final_tfidf = TfidfVectorizer(**best_tfidf_params).fit(X_train)
X_train_tfidf = final_tfidf.transform(X_train).toarray()
X_test_tfidf = final_tfidf.transform(X_test).toarray()

X_train_combined = np.hstack([X_train_tfidf, X_train_lex])
X_test_combined = np.hstack([X_test_tfidf, X_test_lex])

final_model = LogisticRegression(
    C=best_C, class_weight='balanced', random_state=42, max_iter=2000
).fit(X_train_combined, y_train)

print("\n--- Evaluating Final Model (TF-IDF + Lexicon) on Test Set ---")
y_pred = final_model.predict(X_test_combined)
print(classification_report(y_test, y_pred))

Loading data...
Precomputing lexicon features...
Starting Grid Search on TF-IDF + classifier (lexicon features held fixed)...
Fitting 5 folds for each of 36 candidates, totalling 180 fits

Best CV Macro-F1 (TF-IDF only): 0.6933
Winning TF-IDF/clf params: {'clf__C': 10.0, 'tfidf__max_features': 5000, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2)}

--- Evaluating Final Model (TF-IDF + Lexicon) on Test Set ---
              precision    recall  f1-score   support

           1       0.78      0.86      0.82       420
           2       0.75      0.69      0.72       308
           3       0.52      0.49      0.50       141
           4       0.78      0.70      0.74       111

    accuracy                           0.74       980
   macro avg       0.71      0.69      0.70       980
weighted avg       0.73      0.74      0.73       980



In [ ]:
#block_2
import pandas as pd
import re

# 1. We will find 5 posts that actually have at least 1 match
matched_indices = []
raw_features = []

for idx, text in enumerate(X_train):
    # Clean the text exactly like the pipeline does
    text_clean = " " + re.sub(r'[।\.,\?!;:"\'\(\)\[\]\{\}]', '', text).strip() + " "

    # Calculate raw counts for this specific post
    cat_counts = [sum(1 for phrase in cat_phrases[cat] if f" {phrase} " in text_clean) for cat in categories]
    total = sum(cat_counts)

    # If the post triggered our lexicon, save it for the visual test!
    if total > 0:
        matched_indices.append(idx)
        raw_features.append(cat_counts + [total])

    if len(matched_indices) == 5: # Stop after finding 5 good examples
        break

# 2. Put it in a beautiful table to visualize
col_names = list(categories) + ["TOTAL_MATCHES"]
debug_df = pd.DataFrame(raw_features, columns=col_names)

# Add the actual text as the first column so you can read it
debug_df.insert(0, "Original_Post", X_train[matched_indices])

# Display the table!
from IPython.display import display
display(debug_df)

,Original_Post,Somatic Distress & Physical Complaints,Psychiatric Distress & Depressed Affect,"Guilt, Failure & Worthlessness",Anhedonia & Social Withdrawal,Cognitive Impairment & Concentration,"Crisis, Death & Psychache",TOTAL_MATCHES
0,"আসসালামু আলাইকুম, আমি কিছুদিন যাবত অনেক ...",0,2,1,0,0,1,4
1,আমার নাম জুঁই বয়স বিশ বছর আমি বেশ কিছুদিন ধরে...,0,1,0,0,0,0,1
2,ভীষণ ডিপ্রেশন এ থাকি সবসময়। অথচ ডিপ্রেশনে থাকা...,2,6,0,0,0,0,8
3,আপু এই দুইদিন থেকে কেমন যেন আমার মন খারাপ লাগছ...,0,1,0,0,1,0,2
4,আমার বেশির ভাগ সময় মন খারাপ থাকে আর নিজর মধ্যে...,0,2,0,0,0,0,2


In [ ]:
#block_3
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test, y_pred))

[[362  31  25   2]
 [ 57 214  29   8]
 [ 27  33  69  12]
 [ 16   7  10  78]]


In [ ]:
#block_4
import pandas as pd

# 1. Create a quick dataframe mapping the test text to its true and predicted labels
error_analysis_df = pd.DataFrame({
    'Post_Text': X_test_text,
    'True_Label': y_test,
    'Predicted_Label': y_pred_enh  # Make sure this matches your final prediction variable!
})

# 2. Filter strictly for cases where True=3 (Moderate) but Predicted=1 (Control)
error_3_to_1 = error_analysis_df[(error_analysis_df['True_Label'] == 3) & (error_analysis_df['Predicted_Label'] == 1)]

print(f"Found {len(error_3_to_1)} cases where Moderate (3) was under-predicted as Non-depressed (1).")
print("Here are 10 random examples to read for your sanity check:\n")

# 3. Print 10 random examples to read
for idx, row in error_3_to_1.sample(n=min(10, len(error_3_to_1)), random_state=42).iterrows():
    print(f"---")
    print(f"TEXT: {row['Post_Text']}")
    print(f"---")

Found 27 cases where Moderate (3) was under-predicted as Non-depressed (1).
Here are 10 random examples to read for your sanity check:

---
TEXT: আমি ছেলে,বয়স ১৮।কিছুদিন আগে আমার প্যানিক অ্যাটাক হয়,এরপর থেকে আমার কিছু সমস্যা শুরু হয়েছে।১/আমার কাছে কোনো কাজ করতে ভালো লাগে না ২/আগে বিনোদন পেতাম এমন কাজ করতেও মন চায় না ৩/যৌনশক্তি খুব কমে গেছে ৪/খাবার রুচি নাই ৫/ভোর বেলায় ঘুম ভেঙে যায় ৬/সারাদিন ঘরে বসে থাকি,বাহির হইতে ভালো লাগে না,বন্ধুদের সাথে মিশতেও ভালো লাগে না,খুব একা হয়ে সারাদিন ঘরে বসে থাকি ।আমার এই সমস্যা হওয়ার কারণ এবং এথেকে মুক্তির পথ জানতে চাই।ধন্যবাদ
---
---
TEXT: আসলে এমনটি নয় যে আমি এরকম কিছু করতে চাচ্ছি । করার চিন্তা আসছে এটা বলেছি । আর আমি তো জানি এ কষ্ট সাময়িক । কিন্তু কিছুতেই যেহেতু নিয়ন্ত্রণে আসছে না, শারীরিক অবনতি হচ্ছে, তাই জানতে চাচ্ছিলাম ওষুধের সাহায্য নিবো কিনা
---
---
TEXT: আমার কিছু ভালো লাগেনা। চাকুরী করতে ভালো লাগেনা। কর্মসন্তুস্টি আসেনা। আজকাল মনের ওপর নিয়ন্ত্রন হারাচ্ছি। মেজাজ অনেক গরম হয়। তখন হাতের কাছে যা থাকে তাই ভাঙতে থাকি। সুপরামর্শ চাই।
---
---
TEXT: যাকে

In [ ]:
#block_5
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix
from sentence_transformers import SentenceTransformer

print("Loading data...")
train_df = pd.read_csv("train_dataset.csv").dropna(subset=['posts', 'labels'])
test_df = pd.read_csv("test_dataset.csv").dropna(subset=['posts', 'labels'])
lexicon_df = pd.read_excel("/content/Bangla_Depression_Lexicon_latest.xlsx").dropna(
    subset=['term_surface', 'category_name']
)

X_train = train_df['posts'].astype(str).values
y_train = train_df['labels'].astype(int).values
X_test = test_df['posts'].astype(str).values
y_test = test_df['labels'].astype(int).values

categories = sorted(lexicon_df['category_name'].unique())
cat_phrases = {
    cat: lexicon_df[lexicon_df['category_name'] == cat]['term_surface'].astype(str).tolist()
    for cat in categories
}

has_severity_weight = 'severity_association' in lexicon_df.columns
if has_severity_weight:
    cat_weights = {
        cat: dict(zip(
            lexicon_df[lexicon_df['category_name'] == cat]['term_surface'].astype(str),
            lexicon_df[lexicon_df['category_name'] == cat]['severity_association'].astype(float)
        ))
        for cat in categories
    }

PUNCT_RE = re.compile(r'[।\.,\?!;:"\'\(\)\[\]\{\}/]')
# Bangla/ASCII sentence delimiters + numbered-list markers ("১/", "2.") that
# run directly into text without a space, so they need their own split rule.
SEGMENT_SPLIT_RE = re.compile(r'[।\n]|[০-৯0-9]+[/\.]')


def clean_text(text):
    # Replace with a space, not '', so punctuation doesn't fuse adjacent words
    return " " + PUNCT_RE.sub(' ', text).strip() + " "


def split_segments(text):
    segments = [s.strip() for s in SEGMENT_SPLIT_RE.split(text) if s.strip()]
    return segments if segments else [text.strip()]


# ---------------------------------------------------------------------------
# Block 1: exact-match lexicon features (counts, proportions, density,
# severity-weighted score). Scaler is fit once on train and reused at
# transform time — never refit on whatever data is passed in.
# ---------------------------------------------------------------------------
class LexiconFeatureExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        raw = self._raw_features(X)
        self.scaler_ = MinMaxScaler().fit(raw)
        return self

    def transform(self, X):
        raw = self._raw_features(X)
        return self.scaler_.transform(raw)

    def _raw_features(self, X):
        features = []
        for text in X:
            text_clean = clean_text(text)
            word_count = max(len(text.split()), 1)

            cat_counts, cat_weighted = [], []
            for cat in categories:
                phrases = cat_phrases[cat]
                count = sum(1 for phrase in phrases if f" {phrase} " in text_clean)
                cat_counts.append(count)
                if has_severity_weight:
                    weighted = sum(
                        cat_weights[cat].get(phrase, 0.0)
                        for phrase in phrases
                        if f" {phrase} " in text_clean
                    )
                    cat_weighted.append(weighted)

            total = sum(cat_counts)
            row = []
            row += cat_counts
            row += [total]
            row += [(c / total) if total > 0 else 0.0 for c in cat_counts]
            row += [c / word_count for c in cat_counts]
            if has_severity_weight:
                row += cat_weighted
            features.append(row)
        return np.array(features)


# ---------------------------------------------------------------------------
# Block 2: semantic lexicon features. For each document, splits into
# segments, embeds each segment, and takes MAX cosine similarity to each
# category's term embeddings — catches register variation (formal/clinical
# phrasing) that exact-match misses entirely, without diluting long
# multi-symptom posts the way a single whole-document embedding would.
# ---------------------------------------------------------------------------
class SemanticLexiconFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, cat_phrases, model_name='paraphrase-multilingual-mpnet-base-v2'):
        self.cat_phrases = cat_phrases
        self.model_name = model_name

    def fit(self, X, y=None):
        self.model_ = SentenceTransformer(self.model_name)
        self.categories_ = sorted(self.cat_phrases.keys())
        self.term_embeds_ = {
            cat: self.model_.encode(self.cat_phrases[cat], normalize_embeddings=True)
            for cat in self.categories_
        }
        raw = self._raw_features(X)
        self.scaler_ = MinMaxScaler().fit(raw)
        return self

    def transform(self, X):
        raw = self._raw_features(X)
        return self.scaler_.transform(raw)

    def _raw_features(self, X):
        features = []
        for text in X:
            segments = split_segments(str(text))
            seg_embeds = self.model_.encode(segments, normalize_embeddings=True)

            row = []
            for cat in self.categories_:
                sims = seg_embeds @ self.term_embeds_[cat].T
                max_sim = sims.max() if sims.size else 0.0
                top3_mean = np.sort(sims.flatten())[-3:].mean() if sims.size else 0.0
                row += [max_sim, top3_mean]
            features.append(row)
        return np.array(features)


# ---------------------------------------------------------------------------
# Precompute both lexicon feature blocks once, outside the grid search —
# neither depends on any grid parameter, so recomputing them per fold would
# just waste time.
# ---------------------------------------------------------------------------
print("Precomputing exact-match lexicon features...")
lex_extractor = LexiconFeatureExtractor().fit(X_train)
X_train_lex = lex_extractor.transform(X_train)
X_test_lex = lex_extractor.transform(X_test)

print("Precomputing semantic lexicon features (this loads a sentence-transformer model)...")
sem_extractor = SemanticLexiconFeatureExtractor(cat_phrases).fit(X_train)
X_train_sem = sem_extractor.transform(X_train)
X_test_sem = sem_extractor.transform(X_test)

# ---------------------------------------------------------------------------
# Grid search over TF-IDF + classifier only (text signal), same as before.
# ---------------------------------------------------------------------------
tfidf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(class_weight='balanced', random_state=42, max_iter=2000))
])

param_grid = {
    'tfidf__max_features': [3000, 5000, 10000],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df': [2, 5],
    'clf__C': [0.1, 1.0, 10.0]
}

print("Starting Grid Search on TF-IDF + classifier (lexicon features held fixed)...")
grid_search = GridSearchCV(
    tfidf_pipeline, param_grid, cv=5, scoring='f1_macro', n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)
print(f"\nBest CV Macro-F1 (TF-IDF only): {grid_search.best_score_:.4f}")
print(f"Winning TF-IDF/clf params: {grid_search.best_params_}")

best_tfidf_params = {
    k.replace('tfidf__', ''): v for k, v in grid_search.best_params_.items() if k.startswith('tfidf__')
}
best_C = grid_search.best_params_['clf__C']

final_tfidf = TfidfVectorizer(**best_tfidf_params).fit(X_train)
X_train_tfidf = final_tfidf.transform(X_train).toarray()
X_test_tfidf = final_tfidf.transform(X_test).toarray()

# ---------------------------------------------------------------------------
# Three conditions for a clean ablation: TF-IDF only (already have this from
# grid_search.best_score_), TF-IDF + exact-match lexicon, TF-IDF + exact +
# semantic. Report all three so the semantic block's marginal contribution
# is visible on its own, not just folded into one final number.
# ---------------------------------------------------------------------------
def train_and_eval(X_train_feats, X_test_feats, label):
    model = LogisticRegression(
        C=best_C, class_weight='balanced', random_state=42, max_iter=2000
    ).fit(X_train_feats, y_train)
    y_pred = model.predict(X_test_feats)
    print(f"\n--- {label} ---")
    print(classification_report(y_test, y_pred))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred))
    return y_pred


X_train_exact = np.hstack([X_train_tfidf, X_train_lex])
X_test_exact = np.hstack([X_test_tfidf, X_test_lex])
train_and_eval(X_train_exact, X_test_exact, "TF-IDF + Exact-Match Lexicon")

X_train_full = np.hstack([X_train_tfidf, X_train_lex, X_train_sem])
X_test_full = np.hstack([X_test_tfidf, X_test_lex, X_test_sem])
train_and_eval(X_train_full, X_test_full, "TF-IDF + Exact-Match + Semantic Lexicon")

Loading data...
Precomputing exact-match lexicon features...
Precomputing semantic lexicon features (this loads a sentence-transformer model)...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Starting Grid Search on TF-IDF + classifier (lexicon features held fixed)...
Fitting 5 folds for each of 36 candidates, totalling 180 fits

Best CV Macro-F1 (TF-IDF only): 0.6903
Winning TF-IDF/clf params: {'clf__C': 10.0, 'tfidf__max_features': 10000, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2)}

--- TF-IDF + Exact-Match Lexicon ---
              precision    recall  f1-score   support

           1       0.79      0.85      0.82       420
           2       0.75      0.71      0.73       308
           3       0.52      0.49      0.51       141
           4       0.78      0.69      0.73       111

    accuracy                           0.74       980
   macro avg       0.71      0.69      0.70       980
weighted avg       0.73      0.74      0.74       980

Confusion matrix:
[[359  32  26   3]
 [ 56 218  26   8]
 [ 27  34  69  11]
 [ 15   8  11  77]]

--- TF-IDF + Exact-Match + Semantic Lexicon ---
              precision    recall  f1-score   support

           1       0.79  

array([1, 2, 2, 2, 4, 1, 3, 2, 1, 1, 4, 1, 1, 1, 1, 1, 1, 3, 2, 2, 2, 4,
       1, 2, 2, 3, 3, 2, 1, 2, 4, 2, 2, 4, 2, 3, 1, 4, 1, 2, 2, 1, 4, 1,
       1, 1, 2, 2, 2, 2, 2, 1, 4, 3, 2, 3, 2, 1, 1, 3, 2, 2, 1, 1, 1, 2,
       1, 3, 1, 1, 1, 2, 1, 1, 2, 1, 2, 3, 2, 2, 1, 1, 3, 1, 1, 4, 1, 1,
       1, 1, 4, 3, 1, 1, 1, 2, 3, 1, 1, 1, 1, 2, 1, 2, 1, 2, 3, 3, 1, 1,
       1, 1, 1, 2, 1, 4, 1, 2, 2, 1, 2, 1, 1, 1, 2, 1, 2, 1, 2, 1, 1, 2,
       1, 2, 3, 4, 1, 1, 1, 1, 4, 3, 1, 1, 1, 1, 1, 1, 4, 1, 1, 3, 1, 1,
       2, 1, 3, 1, 1, 2, 1, 4, 1, 3, 1, 1, 1, 1, 2, 1, 1, 1, 1, 3, 4, 1,
       1, 2, 4, 1, 2, 1, 4, 1, 2, 2, 3, 4, 2, 3, 1, 4, 2, 2, 2, 1, 2, 1,
       4, 1, 1, 1, 2, 1, 4, 1, 2, 3, 1, 1, 2, 2, 1, 2, 1, 2, 1, 4, 1, 1,
       2, 2, 1, 4, 2, 1, 2, 4, 1, 2, 1, 3, 3, 1, 1, 2, 4, 4, 2, 2, 3, 1,
       3, 3, 4, 1, 2, 4, 1, 1, 4, 2, 3, 2, 1, 1, 2, 2, 1, 1, 2, 1, 2, 3,
       1, 2, 1, 1, 3, 4, 2, 1, 1, 1, 1, 1, 2, 1, 2, 1, 1, 1, 1, 1, 2, 2,
       2, 3, 1, 4, 1, 4, 1, 1, 2, 2, 2, 4, 1, 1, 1,

In [ ]:
#block_7
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix
from sentence_transformers import SentenceTransformer

print("Loading data...")
train_df = pd.read_csv("train_dataset.csv").dropna(subset=['posts', 'labels'])
test_df = pd.read_csv("test_dataset.csv").dropna(subset=['posts', 'labels'])
lexicon_df = pd.read_excel("/content/Bangla_Depression_Lexicon_latest.xlsx").dropna(
    subset=['term_surface', 'category_name']
)

X_train = train_df['posts'].astype(str).values
y_train = train_df['labels'].astype(int).values
X_test = test_df['posts'].astype(str).values
y_test = test_df['labels'].astype(int).values

# ---------------------------------------------------------------------------
# CUSTOM CLASS WEIGHTS: This is our mathematical steering wheel.
# We explicitly force the model to prioritize the minority/difficult classes.
# ---------------------------------------------------------------------------
custom_weights = {1: 1.0, 2: 2.0, 3: 4.0, 4: 2.5}

categories = sorted(lexicon_df['category_name'].unique())
cat_phrases = {
    cat: lexicon_df[lexicon_df['category_name'] == cat]['term_surface'].astype(str).tolist()
    for cat in categories
}

has_severity_weight = 'severity_association' in lexicon_df.columns
if has_severity_weight:
    cat_weights = {
        cat: dict(zip(
            lexicon_df[lexicon_df['category_name'] == cat]['term_surface'].astype(str),
            lexicon_df[lexicon_df['category_name'] == cat]['severity_association'].astype(float)
        ))
        for cat in categories
    }

PUNCT_RE = re.compile(r'[।\.,\?!;:"\'\(\)\[\]\{\}/]')
# Bangla/ASCII sentence delimiters + numbered-list markers ("১/", "2.") that
# run directly into text without a space, so they need their own split rule.
SEGMENT_SPLIT_RE = re.compile(r'[।\n]|[০-৯0-9]+[/\.]')

def clean_text(text):
    # Replace with a space, not '', so punctuation doesn't fuse adjacent words
    return " " + PUNCT_RE.sub(' ', text).strip() + " "

def split_segments(text):
    segments = [s.strip() for s in SEGMENT_SPLIT_RE.split(text) if s.strip()]
    return segments if segments else [text.strip()]

# ---------------------------------------------------------------------------
# Block 1: exact-match lexicon features (counts, proportions, density,
# severity-weighted score). Scaler is fit once on train and reused at
# transform time — never refit on whatever data is passed in.
# ---------------------------------------------------------------------------
class LexiconFeatureExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        raw = self._raw_features(X)
        self.scaler_ = MinMaxScaler().fit(raw)
        return self

    def transform(self, X):
        raw = self._raw_features(X)
        return self.scaler_.transform(raw)

    def _raw_features(self, X):
        features = []
        for text in X:
            text_clean = clean_text(text)
            word_count = max(len(text.split()), 1)

            cat_counts, cat_weighted = [], []
            for cat in categories:
                phrases = cat_phrases[cat]
                count = sum(1 for phrase in phrases if f" {phrase} " in text_clean)
                cat_counts.append(count)
                if has_severity_weight:
                    weighted = sum(
                        cat_weights[cat].get(phrase, 0.0)
                        for phrase in phrases
                        if f" {phrase} " in text_clean
                    )
                    cat_weighted.append(weighted)

            total = sum(cat_counts)
            row = []
            row += cat_counts
            row += [total]
            row += [(c / total) if total > 0 else 0.0 for c in cat_counts]
            row += [c / word_count for c in cat_counts]
            if has_severity_weight:
                row += cat_weighted
            features.append(row)
        return np.array(features)

# ---------------------------------------------------------------------------
# Block 2: semantic lexicon features. For each document, splits into
# segments, embeds each segment, and takes MAX cosine similarity to each
# category's term embeddings — catches register variation (formal/clinical
# phrasing) that exact-match misses entirely, without diluting long
# multi-symptom posts the way a single whole-document embedding would.
# ---------------------------------------------------------------------------
class SemanticLexiconFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, cat_phrases, model_name='paraphrase-multilingual-mpnet-base-v2'):
        self.cat_phrases = cat_ph

Loading data...


In [ ]:
#block_8
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix
from sentence_transformers import SentenceTransformer

print("Loading data...")
train_df = pd.read_csv("train_dataset.csv").dropna(subset=['posts', 'labels'])
test_df = pd.read_csv("test_dataset.csv").dropna(subset=['posts', 'labels'])
lexicon_df = pd.read_excel("/content/Bangla_Depression_Lexicon_latest.xlsx").dropna(
    subset=['term_surface', 'category_name']
)

X_train = train_df['posts'].astype(str).values
y_train = train_df['labels'].astype(int).values
X_test = test_df['posts'].astype(str).values
y_test = test_df['labels'].astype(int).values

# ---------------------------------------------------------------------------
# CUSTOM CLASS WEIGHTS: This is our mathematical steering wheel.
# We explicitly force the model to prioritize the minority/difficult classes.
# ---------------------------------------------------------------------------
custom_weights = {1: 1.0, 2: 2.0, 3: 4.0, 4: 2.5}

categories = sorted(lexicon_df['category_name'].unique())
cat_phrases = {
    cat: lexicon_df[lexicon_df['category_name'] == cat]['term_surface'].astype(str).tolist()
    for cat in categories
}

has_severity_weight = 'severity_association' in lexicon_df.columns
if has_severity_weight:
    cat_weights = {
        cat: dict(zip(
            lexicon_df[lexicon_df['category_name'] == cat]['term_surface'].astype(str),
            lexicon_df[lexicon_df['category_name'] == cat]['severity_association'].astype(float)
        ))
        for cat in categories
    }

PUNCT_RE = re.compile(r'[।\.,\?!;:"\'\(\)\[\]\{\}/]')
# Bangla/ASCII sentence delimiters + numbered-list markers ("১/", "2.") that
# run directly into text without a space, so they need their own split rule.
SEGMENT_SPLIT_RE = re.compile(r'[।\n]|[০-৯0-9]+[/\.]')

def clean_text(text):
    # Replace with a space, not '', so punctuation doesn't fuse adjacent words
    return " " + PUNCT_RE.sub(' ', text).strip() + " "

def split_segments(text):
    segments = [s.strip() for s in SEGMENT_SPLIT_RE.split(text) if s.strip()]
    return segments if segments else [text.strip()]

# ---------------------------------------------------------------------------
# Block 1: exact-match lexicon features (counts, proportions, density,
# severity-weighted score). Scaler is fit once on train and reused at
# transform time — never refit on whatever data is passed in.
# ---------------------------------------------------------------------------
class LexiconFeatureExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        raw = self._raw_features(X)
        self.scaler_ = MinMaxScaler().fit(raw)
        return self

    def transform(self, X):
        raw = self._raw_features(X)
        return self.scaler_.transform(raw)

    def _raw_features(self, X):
        features = []
        for text in X:
            text_clean = clean_text(text)
            word_count = max(len(text.split()), 1)

            cat_counts, cat_weighted = [], []
            for cat in categories:
                phrases = cat_phrases[cat]
                count = sum(1 for phrase in phrases if f" {phrase} " in text_clean)
                cat_counts.append(count)
                if has_severity_weight:
                    weighted = sum(
                        cat_weights[cat].get(phrase, 0.0)
                        for phrase in phrases
                        if f" {phrase} " in text_clean
                    )
                    cat_weighted.append(weighted)

            total = sum(cat_counts)
            row = []
            row += cat_counts
            row += [total]
            row += [(c / total) if total > 0 else 0.0 for c in cat_counts]
            row += [c / word_count for c in cat_counts]
            if has_severity_weight:
                row += cat_weighted
            features.append(row)
        return np.array(features)

# ---------------------------------------------------------------------------
# Block 2: semantic lexicon features. For each document, splits into
# segments, embeds each segment, and takes MAX cosine similarity to each
# category's term embeddings — catches register variation (formal/clinical
# phrasing) that exact-match misses entirely, without diluting long
# multi-symptom posts the way a single whole-document embedding would.
# ---------------------------------------------------------------------------
class SemanticLexiconFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, cat_phrases, model_name='paraphrase-multilingual-mpnet-base-v2'):
        self.cat_phrases = cat_phrases
        self.model_name = model_name

    def fit(self, X, y=None):
        self.model_ = SentenceTransformer(self.model_name)
        self.categories_ = sorted(self.cat_phrases.keys())
        self.term_embeds_ = {
            cat: self.model_.encode(self.cat_phrases[cat], normalize_embeddings=True)
            for cat in self.categories_
        }
        raw = self._raw_features(X)
        self.scaler_ = MinMaxScaler().fit(raw)
        return self

    def transform(self, X):
        raw = self._raw_features(X)
        return self.scaler_.transform(raw)

    def _raw_features(self, X):
        features = []
        for text in X:
            segments = split_segments(str(text))
            seg_embeds = self.model_.encode(segments, normalize_embeddings=True)

            row = []
            for cat in self.categories_:
                sims = seg_embeds @ self.term_embeds_[cat].T
                max_sim = sims.max() if sims.size else 0.0
                top3_mean = np.sort(sims.flatten())[-3:].mean() if sims.size else 0.0
                row += [max_sim, top3_mean]
            features.append(row)
        return np.array(features)

# ---------------------------------------------------------------------------
# Precompute both lexicon feature blocks once, outside the grid search —
# neither depends on any grid parameter, so recomputing them per fold would
# just waste time.
# ---------------------------------------------------------------------------
print("Precomputing exact-match lexicon features...")
lex_extractor = LexiconFeatureExtractor().fit(X_train)
X_train_lex = lex_extractor.transform(X_train)
X_test_lex = lex_extractor.transform(X_test)

print("Precomputing semantic lexicon features (this loads a sentence-transformer model)...")
sem_extractor = SemanticLexiconFeatureExtractor(cat_phrases).fit(X_train)
X_train_sem = sem_extractor.transform(X_train)
X_test_sem = sem_extractor.transform(X_test)

# ---------------------------------------------------------------------------
# Grid search over TF-IDF + classifier only (text signal), same as before.
# Note the use of custom_weights!
# ---------------------------------------------------------------------------
tfidf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(class_weight=custom_weights, random_state=42, max_iter=2000))
])

param_grid = {
    'tfidf__max_features': [3000, 5000, 10000],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df': [2, 5],
    'clf__C': [0.1, 1.0, 10.0]
}

print("\nStarting Grid Search on TF-IDF + classifier (lexicon features held fixed)...")
grid_search = GridSearchCV(
    tfidf_pipeline, param_grid, cv=5, scoring='f1_macro', n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)
print(f"\nBest CV Macro-F1 (TF-IDF only): {grid_search.best_score_:.4f}")
print(f"Winning TF-IDF/clf params: {grid_search.best_params_}")

best_tfidf_params = {
    k.replace('tfidf__', ''): v for k, v in grid_search.best_params_.items() if k.startswith('tfidf__')
}
best_C = grid_search.best_params_['clf__C']

final_tfidf = TfidfVectorizer(**best_tfidf_params).fit(X_train)
X_train_tfidf = final_tfidf.transform(X_train).toarray()
X_test_tfidf = final_tfidf.transform(X_test).toarray()

# ---------------------------------------------------------------------------
# Three conditions for a clean ablation: TF-IDF only, TF-IDF + exact-match,
# and TF-IDF + exact + semantic.
# ---------------------------------------------------------------------------
def train_and_eval(X_train_feats, X_test_feats, label):
    # Note the use of custom_weights here as well!
    model = LogisticRegression(
        C=best_C, class_weight=custom_weights, random_state=42, max_iter=2000
    ).fit(X_train_feats, y_train)

    y_pred = model.predict(X_test_feats)
    print(f"\n=======================================================")
    print(f"--- {label} ---")
    print(f"=======================================================")
    print(classification_report(y_test, y_pred))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred))
    return y_pred


X_train_exact = np.hstack([X_train_tfidf, X_train_lex])
X_test_exact = np.hstack([X_test_tfidf, X_test_lex])

# Run all 3 Ablations:
print("\n[Running Final Evaluations on Locked Test Set]")
train_and_eval(X_train_tfidf, X_test_tfidf, "TF-IDF ONLY (Baseline)")
train_and_eval(X_train_exact, X_test_exact, "TF-IDF + Exact-Match Lexicon")

X_train_full = np.hstack([X_train_tfidf, X_train_lex, X_train_sem])
X_test_full = np.hstack([X_test_tfidf, X_test_lex, X_test_sem])
train_and_eval(X_train_full, X_test_full, "TF-IDF + Exact-Match + Semantic Lexicon")


Loading data...
Precomputing exact-match lexicon features...
Precomputing semantic lexicon features (this loads a sentence-transformer model)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Starting Grid Search on TF-IDF + classifier (lexicon features held fixed)...
Fitting 5 folds for each of 36 candidates, totalling 180 fits

Best CV Macro-F1 (TF-IDF only): 0.6847
Winning TF-IDF/clf params: {'clf__C': 10.0, 'tfidf__max_features': 10000, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2)}

[Running Final Evaluations on Locked Test Set]

--- TF-IDF ONLY (Baseline) ---
              precision    recall  f1-score   support

           1       0.77      0.81      0.79       420
           2       0.70      0.71      0.71       308
           3       0.53      0.50      0.51       141
           4       0.83      0.68      0.75       111

    accuracy                           0.72       980
   macro avg       0.71      0.67      0.69       980
weighted avg       0.72      0.72      0.72       980

Confusion matrix:
[[342  44  28   6]
 [ 61 219  25   3]
 [ 26  39  70   6]
 [ 16  11   9  75]]

--- TF-IDF + Exact-Match Lexicon ---
              precision    recall  f1-score   su

array([1, 2, 2, 2, 4, 1, 2, 2, 1, 1, 4, 1, 2, 1, 1, 1, 1, 3, 2, 2, 2, 4,
       1, 2, 2, 3, 3, 2, 1, 2, 4, 2, 2, 4, 2, 3, 1, 1, 1, 2, 2, 1, 4, 1,
       1, 1, 2, 2, 2, 2, 2, 1, 4, 3, 2, 3, 2, 1, 1, 3, 2, 2, 1, 1, 1, 2,
       1, 3, 1, 1, 1, 2, 1, 1, 2, 1, 2, 3, 2, 2, 1, 1, 3, 1, 1, 1, 1, 1,
       1, 1, 4, 3, 1, 1, 1, 2, 3, 1, 1, 1, 1, 2, 1, 2, 1, 2, 3, 3, 1, 1,
       1, 1, 2, 2, 1, 4, 1, 2, 2, 1, 2, 1, 1, 1, 2, 1, 2, 1, 2, 1, 1, 2,
       1, 2, 3, 4, 3, 1, 1, 1, 4, 3, 1, 1, 1, 1, 1, 1, 4, 1, 1, 3, 1, 1,
       2, 1, 3, 1, 1, 2, 1, 4, 1, 3, 1, 1, 1, 1, 2, 1, 1, 1, 1, 3, 4, 1,
       1, 2, 1, 1, 2, 1, 4, 1, 2, 2, 3, 4, 2, 3, 1, 4, 2, 2, 2, 1, 2, 1,
       4, 1, 1, 1, 2, 1, 4, 1, 2, 3, 1, 1, 2, 2, 1, 2, 1, 2, 1, 4, 1, 1,
       2, 2, 1, 4, 2, 1, 2, 4, 1, 2, 1, 3, 3, 1, 1, 2, 4, 4, 2, 2, 3, 1,
       3, 3, 4, 1, 2, 4, 1, 1, 4, 2, 3, 2, 1, 1, 2, 2, 1, 1, 2, 1, 2, 3,
       1, 2, 1, 1, 3, 4, 2, 1, 1, 1, 1, 1, 2, 1, 2, 1, 1, 1, 1, 1, 2, 2,
       2, 3, 1, 4, 1, 4, 1, 1, 2, 2, 2, 4, 1, 1, 1,

In [ ]:
#block_9
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix
from sentence_transformers import SentenceTransformer

print("Loading data...")
train_df = pd.read_csv("train_dataset.csv").dropna(subset=['posts', 'labels'])
test_df = pd.read_csv("test_dataset.csv").dropna(subset=['posts', 'labels'])
lexicon_df = pd.read_excel("/content/Bangla_Depression_Lexicon_latest.xlsx").dropna(
    subset=['term_surface', 'category_name']
)

X_train = train_df['posts'].astype(str).values
y_train = train_df['labels'].astype(int).values
X_test = test_df['posts'].astype(str).values
y_test = test_df['labels'].astype(int).values

# ---------------------------------------------------------------------------
# CUSTOM CLASS WEIGHTS: This is our mathematical steering wheel.
# We explicitly force the model to prioritize the minority/difficult classes.
# ---------------------------------------------------------------------------
custom_weights = {1: 1.0, 2: 2.0, 3: 4.0, 4: 2.5}

categories = sorted(lexicon_df['category_name'].unique())
cat_phrases = {
    cat: lexicon_df[lexicon_df['category_name'] == cat]['term_surface'].astype(str).tolist()
    for cat in categories
}

has_severity_weight = 'severity_association' in lexicon_df.columns
if has_severity_weight:
    cat_weights = {
        cat: dict(zip(
            lexicon_df[lexicon_df['category_name'] == cat]['term_surface'].astype(str),
            lexicon_df[lexicon_df['category_name'] == cat]['severity_association'].astype(float)
        ))
        for cat in categories
    }

PUNCT_RE = re.compile(r'[।\.,\?!;:"\'\(\)\[\]\{\}/]')
# Bangla/ASCII sentence delimiters + numbered-list markers ("১/", "2.") that
# run directly into text without a space, so they need their own split rule.
SEGMENT_SPLIT_RE = re.compile(r'[।\n]|[০-৯0-9]+[/\.]')

def clean_text(text):
    # Replace with a space, not '', so punctuation doesn't fuse adjacent words
    return " " + PUNCT_RE.sub(' ', text).strip() + " "

def split_segments(text):
    segments = [s.strip() for s in SEGMENT_SPLIT_RE.split(text) if s.strip()]
    return segments if segments else [text.strip()]

# ---------------------------------------------------------------------------
# Block 1: exact-match lexicon features (counts, proportions, density,
# severity-weighted score). Scaler is fit once on train and reused at
# transform time — never refit on whatever data is passed in.
# ---------------------------------------------------------------------------
class LexiconFeatureExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        raw = self._raw_features(X)
        self.scaler_ = MinMaxScaler().fit(raw)
        return self

    def transform(self, X):
        raw = self._raw_features(X)
        return self.scaler_.transform(raw)

    def _raw_features(self, X):
        features = []
        for text in X:
            text_clean = clean_text(text)
            word_count = max(len(text.split()), 1)

            cat_counts, cat_weighted = [], []
            for cat in categories:
                phrases = cat_phrases[cat]
                count = sum(1 for phrase in phrases if f" {phrase} " in text_clean)
                cat_counts.append(count)
                if has_severity_weight:
                    weighted = sum(
                        cat_weights[cat].get(phrase, 0.0)
                        for phrase in phrases
                        if f" {phrase} " in text_clean
                    )
                    cat_weighted.append(weighted)

            total = sum(cat_counts)
            row = []
            row += cat_counts
            row += [total]
            row += [(c / total) if total > 0 else 0.0 for c in cat_counts]
            row += [c / word_count for c in cat_counts]
            if has_severity_weight:
                row += cat_weighted
            features.append(row)
        return np.array(features)

# ---------------------------------------------------------------------------
# Block 2: semantic lexicon features. For each document, splits into
# segments, embeds each segment, and takes MAX cosine similarity to each
# category's term embeddings — catches register variation (formal/clinical
# phrasing) that exact-match misses entirely, without diluting long
# multi-symptom posts the way a single whole-document embedding would.
# ---------------------------------------------------------------------------
class SemanticLexiconFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, cat_phrases, model_name='paraphrase-multilingual-mpnet-base-v2'):
        self.cat_phrases = cat_phrases
        self.model_name = model_name

    def fit(self, X, y=None):
        self.model_ = SentenceTransformer(self.model_name)
        self.categories_ = sorted(self.cat_phrases.keys())
        self.term_embeds_ = {
            cat: self.model_.encode(self.cat_phrases[cat], normalize_embeddings=True)
            for cat in self.categories_
        }
        raw = self._raw_features(X)
        self.scaler_ = MinMaxScaler().fit(raw)
        return self

    def transform(self, X):
        raw = self._raw_features(X)
        return self.scaler_.transform(raw)

    def _raw_features(self, X):
        features = []
        for text in X:
            text_clean = clean_text(text)
            word_count = max(len(text.split()), 1)

            cat_counts, cat_weighted = [], []
            for cat in categories:
                phrases = cat_phrases[cat]
                count = sum(1 for phrase in phrases if f" {phrase} " in text_clean)
                cat_counts.append(count)
                if has_severity_weight:
                    weighted = sum(
                        cat_weights[cat].get(phrase, 0.0)
                        for phrase in phrases
                        if f" {phrase} " in text_clean
                    )
                    cat_weighted.append(weighted)

            total = sum(cat_counts)

            # --- THE NEW CLINICAL BREADTH FEATURE ---
            # Counts how many distinct categories had at least 1 hit (0 to 6)
            breadth_score = sum(1 for c in cat_counts if c > 0)

            row = []
            row += cat_counts
            row += [total]
            row += [breadth_score]  # <-- Added to the matrix here!
            row += [(c / total) if total > 0 else 0.0 for c in cat_counts]
            row += [c / word_count for c in cat_counts]
            if has_severity_weight:
                row += cat_weighted
            features.append(row)
        return np.array(features)

# ---------------------------------------------------------------------------
# Precompute both lexicon feature blocks once, outside the grid search —
# neither depends on any grid parameter, so recomputing them per fold would
# just waste time.
# ---------------------------------------------------------------------------
print("Precomputing exact-match lexicon features...")
lex_extractor = LexiconFeatureExtractor().fit(X_train)
X_train_lex = lex_extractor.transform(X_train)
X_test_lex = lex_extractor.transform(X_test)

print("Precomputing semantic lexicon features (this loads a sentence-transformer model)...")
sem_extractor = SemanticLexiconFeatureExtractor(cat_phrases).fit(X_train)
X_train_sem = sem_extractor.transform(X_train)
X_test_sem = sem_extractor.transform(X_test)

# ---------------------------------------------------------------------------
# Grid search over TF-IDF + classifier only (text signal), same as before.
# Note the use of custom_weights!
# ---------------------------------------------------------------------------
tfidf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(class_weight=custom_weights, random_state=42, max_iter=2000))
])

param_grid = {
    'tfidf__max_features': [3000, 5000, 10000],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df': [2, 5],
    'clf__C': [0.1, 1.0, 10.0]
}

print("\nStarting Grid Search on TF-IDF + classifier (lexicon features held fixed)...")
grid_search = GridSearchCV(
    tfidf_pipeline, param_grid, cv=5, scoring='f1_macro', n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)
print(f"\nBest CV Macro-F1 (TF-IDF only): {grid_search.best_score_:.4f}")
print(f"Winning TF-IDF/clf params: {grid_search.best_params_}")

best_tfidf_params = {
    k.replace('tfidf__', ''): v for k, v in grid_search.best_params_.items() if k.startswith('tfidf__')
}
best_C = grid_search.best_params_['clf__C']

final_tfidf = TfidfVectorizer(**best_tfidf_params).fit(X_train)
X_train_tfidf = final_tfidf.transform(X_train).toarray()
X_test_tfidf = final_tfidf.transform(X_test).toarray()

# ---------------------------------------------------------------------------
# Three conditions for a clean ablation: TF-IDF only, TF-IDF + exact-match,
# and TF-IDF + exact + semantic.
# ---------------------------------------------------------------------------
def train_and_eval(X_train_feats, X_test_feats, label):
    # Note the use of custom_weights here as well!
    model = LogisticRegression(
        C=best_C, class_weight=custom_weights, random_state=42, max_iter=2000
    ).fit(X_train_feats, y_train)

    y_pred = model.predict(X_test_feats)
    print(f"\n=======================================================")
    print(f"--- {label} ---")
    print(f"=======================================================")
    print(classification_report(y_test, y_pred))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred))
    return y_pred


X_train_exact = np.hstack([X_train_tfidf, X_train_lex])
X_test_exact = np.hstack([X_test_tfidf, X_test_lex])

# Run all 3 Ablations:
print("\n[Running Final Evaluations on Locked Test Set]")
train_and_eval(X_train_tfidf, X_test_tfidf, "TF-IDF ONLY (Baseline)")
train_and_eval(X_train_exact, X_test_exact, "TF-IDF + Exact-Match Lexicon")

X_train_full = np.hstack([X_train_tfidf, X_train_lex, X_train_sem])
X_test_full = np.hstack([X_test_tfidf, X_test_lex, X_test_sem])
train_and_eval(X_train_full, X_test_full, "TF-IDF + Exact-Match + Semantic Lexicon")

Loading data...
Precomputing exact-match lexicon features...
Precomputing semantic lexicon features (this loads a sentence-transformer model)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Starting Grid Search on TF-IDF + classifier (lexicon features held fixed)...
Fitting 5 folds for each of 36 candidates, totalling 180 fits

Best CV Macro-F1 (TF-IDF only): 0.6847
Winning TF-IDF/clf params: {'clf__C': 10.0, 'tfidf__max_features': 10000, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2)}

[Running Final Evaluations on Locked Test Set]

--- TF-IDF ONLY (Baseline) ---
              precision    recall  f1-score   support

           1       0.77      0.81      0.79       420
           2       0.70      0.71      0.71       308
           3       0.53      0.50      0.51       141
           4       0.83      0.68      0.75       111

    accuracy                           0.72       980
   macro avg       0.71      0.67      0.69       980
weighted avg       0.72      0.72      0.72       980

Confusion matrix:
[[342  44  28   6]
 [ 61 219  25   3]
 [ 26  39  70   6]
 [ 16  11   9  75]]

--- TF-IDF + Exact-Match Lexicon ---
              precision    recall  f1-score   su

array([1, 2, 2, 2, 4, 1, 2, 2, 1, 1, 3, 1, 2, 1, 1, 1, 1, 3, 2, 2, 2, 3,
       1, 2, 2, 3, 3, 2, 1, 2, 4, 2, 2, 4, 2, 3, 1, 4, 1, 2, 2, 1, 4, 1,
       1, 1, 2, 2, 2, 2, 2, 1, 3, 3, 2, 3, 2, 1, 1, 1, 2, 2, 1, 1, 1, 2,
       1, 3, 1, 1, 1, 2, 1, 2, 2, 1, 2, 3, 2, 2, 1, 1, 3, 1, 1, 4, 1, 1,
       1, 1, 4, 3, 1, 1, 1, 2, 3, 1, 1, 1, 1, 2, 1, 2, 1, 2, 3, 3, 1, 1,
       1, 1, 2, 2, 1, 4, 1, 2, 2, 1, 2, 1, 1, 1, 2, 1, 2, 1, 2, 1, 1, 2,
       1, 2, 3, 4, 1, 1, 1, 1, 4, 2, 1, 3, 1, 1, 1, 1, 4, 1, 1, 3, 1, 1,
       2, 1, 1, 1, 1, 2, 1, 1, 1, 3, 1, 1, 1, 1, 2, 1, 1, 1, 3, 3, 4, 1,
       1, 2, 1, 1, 2, 1, 4, 1, 2, 2, 3, 4, 2, 3, 1, 4, 2, 2, 2, 1, 2, 1,
       4, 1, 1, 1, 2, 1, 3, 1, 2, 3, 1, 1, 1, 2, 1, 2, 1, 2, 1, 4, 1, 1,
       2, 2, 1, 4, 2, 1, 2, 4, 1, 2, 1, 3, 3, 1, 1, 2, 4, 4, 2, 2, 3, 1,
       3, 3, 4, 3, 2, 4, 1, 1, 4, 2, 3, 2, 1, 1, 2, 2, 1, 1, 2, 1, 2, 3,
       1, 2, 1, 1, 3, 4, 2, 1, 1, 1, 1, 1, 2, 1, 2, 1, 1, 1, 1, 1, 2, 2,
       2, 3, 1, 4, 1, 4, 1, 1, 4, 2, 2, 4, 1, 1, 1,

In [ ]:
#block_10
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix
from sentence_transformers import SentenceTransformer

print("Loading data...")
train_df = pd.read_csv("train_dataset.csv").dropna(subset=['posts', 'labels'])
test_df = pd.read_csv("test_dataset.csv").dropna(subset=['posts', 'labels'])
lexicon_df = pd.read_excel("/content/Bangla_Depression_Lexicon_latest.xlsx").dropna(
    subset=['term_surface', 'category_name']
)

X_train = train_df['posts'].astype(str).values
y_train = train_df['labels'].astype(int).values
X_test = test_df['posts'].astype(str).values
y_test = test_df['labels'].astype(int).values

# ---------------------------------------------------------------------------
# CUSTOM CLASS WEIGHTS: This is our mathematical steering wheel.
# We explicitly force the model to prioritize the minority/difficult classes.
# ---------------------------------------------------------------------------
custom_weights = {1: 1.0, 2: 2.0, 3: 4.0, 4: 2.5}

categories = sorted(lexicon_df['category_name'].unique())
cat_phrases = {
    cat: lexicon_df[lexicon_df['category_name'] == cat]['term_surface'].astype(str).tolist()
    for cat in categories
}

has_severity_weight = 'severity_association' in lexicon_df.columns
if has_severity_weight:
    cat_weights = {
        cat: dict(zip(
            lexicon_df[lexicon_df['category_name'] == cat]['term_surface'].astype(str),
            lexicon_df[lexicon_df['category_name'] == cat]['severity_association'].astype(float)
        ))
        for cat in categories
    }

PUNCT_RE = re.compile(r'[।\.,\?!;:"\'\(\)\[\]\{\}/]')
# Bangla/ASCII sentence delimiters + numbered-list markers ("১/", "2.") that
# run directly into text without a space, so they need their own split rule.
SEGMENT_SPLIT_RE = re.compile(r'[।\n]|[০-৯0-9]+[/\.]')

def clean_text(text):
    # Replace with a space, not '', so punctuation doesn't fuse adjacent words
    return " " + PUNCT_RE.sub(' ', text).strip() + " "

def split_segments(text):
    segments = [s.strip() for s in SEGMENT_SPLIT_RE.split(text) if s.strip()]
    return segments if segments else [text.strip()]

# ---------------------------------------------------------------------------
# Block 1: exact-match lexicon features (counts, proportions, density,
# severity-weighted score). Scaler is fit once on train and reused at
# transform time — never refit on whatever data is passed in.
# ---------------------------------------------------------------------------
class LexiconFeatureExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        raw = self._raw_features(X)
        self.scaler_ = MinMaxScaler().fit(raw)
        return self

    def transform(self, X):
        raw = self._raw_features(X)
        return self.scaler_.transform(raw)

    def _raw_features(self, X):
        features = []
        for text in X:
            text_clean = clean_text(text)
            word_count = max(len(text.split()), 1)

            cat_counts, cat_weighted = [], []
            for cat in categories:
                phrases = cat_phrases[cat]
                count = sum(1 for phrase in phrases if f" {phrase} " in text_clean)
                cat_counts.append(count)
                if has_severity_weight:
                    weighted = sum(
                        cat_weights[cat].get(phrase, 0.0)
                        for phrase in phrases
                        if f" {phrase} " in text_clean
                    )
                    cat_weighted.append(weighted)

            total = sum(cat_counts)
            row = []
            row += cat_counts
            row += [total]
            row += [(c / total) if total > 0 else 0.0 for c in cat_counts]
            row += [c / word_count for c in cat_counts]
            if has_severity_weight:
                row += cat_weighted
            features.append(row)
        return np.array(features)

# ---------------------------------------------------------------------------
# Block 2: semantic lexicon features. For each document, splits into
# segments, embeds each segment, and takes MAX cosine similarity to each
# category's term embeddings — catches register variation (formal/clinical
# phrasing) that exact-match misses entirely, without diluting long
# multi-symptom posts the way a single whole-document embedding would.
# ---------------------------------------------------------------------------
class SemanticLexiconFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, cat_phrases, model_name='paraphrase-multilingual-mpnet-base-v2'):
        self.cat_phrases = cat_phrases
        self.model_name = model_name

    def fit(self, X, y=None):
        self.model_ = SentenceTransformer(self.model_name)
        self.categories_ = sorted(self.cat_phrases.keys())
        self.term_embeds_ = {
            cat: self.model_.encode(self.cat_phrases[cat], normalize_embeddings=True)
            for cat in self.categories_
        }
        raw = self._raw_features(X)
        self.scaler_ = MinMaxScaler().fit(raw)
        return self

    def transform(self, X):
        raw = self._raw_features(X)
        return self.scaler_.transform(raw)

    def _raw_features(self, X):

        features = []
        for text in X:
            text_clean = clean_text(text)
            word_count = max(len(text.split()), 1)

            cat_counts, cat_weighted = [], []
            for cat in categories:
                phrases = cat_phrases[cat]
                count = sum(1 for phrase in phrases if f" {phrase} " in text_clean)
                cat_counts.append(count)
                if has_severity_weight:
                    weighted = sum(
                        cat_weights[cat].get(phrase, 0.0)
                        for phrase in phrases
                        if f" {phrase} " in text_clean
                    )
                    cat_weighted.append(weighted)

            total = sum(cat_counts)

            # --- THE NEW CLINICAL BREADTH FEATURE ---
            # Counts how many distinct categories had at least 1 hit (0 to 6)
            breadth_score = sum(1 for c in cat_counts if c > 0)
                        # ...
            total = sum(cat_counts)
            breadth_score = sum(1 for c in cat_counts if c > 0)

            # --- THE NEW SYMPTOM-FREE FLAG ---
            # Explicitly flags the post as 1 if ZERO lexicon words were found
            is_symptom_free = 1.0 if total == 0 else 0.0

            row = []
            row += cat_counts
            row += [total]
            row += [breadth_score]
            row += [is_symptom_free]  # <-- Add the flag to the matrix here!
            row += [(c / total) if total > 0 else 0.0 for c in cat_counts]
            # ...

            row = []
            row += cat_counts
            row += [total]
            row += [breadth_score]  # <-- Added to the matrix here!
            row += [(c / total) if total > 0 else 0.0 for c in cat_counts]
            row += [c / word_count for c in cat_counts]
            if has_severity_weight:
                row += cat_weighted
            features.append(row)
        return np.array(features)

# ---------------------------------------------------------------------------
# Precompute both lexicon feature blocks once, outside the grid search —
# neither depends on any grid parameter, so recomputing them per fold would
# just waste time.
# ---------------------------------------------------------------------------
print("Precomputing exact-match lexicon features...")
lex_extractor = LexiconFeatureExtractor().fit(X_train)
X_train_lex = lex_extractor.transform(X_train)
X_test_lex = lex_extractor.transform(X_test)

print("Precomputing semantic lexicon features (this loads a sentence-transformer model)...")
sem_extractor = SemanticLexiconFeatureExtractor(cat_phrases).fit(X_train)
X_train_sem = sem_extractor.transform(X_train)
X_test_sem = sem_extractor.transform(X_test)

# ---------------------------------------------------------------------------
# Grid search over TF-IDF + classifier only (text signal), same as before.
# Note the use of custom_weights!
# ---------------------------------------------------------------------------
tfidf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(class_weight=custom_weights, random_state=42, max_iter=2000))
])

param_grid = {
    'tfidf__max_features': [3000, 5000, 10000],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df': [2, 5],
    'clf__C': [0.1, 1.0, 10.0]
}

print("\nStarting Grid Search on TF-IDF + classifier (lexicon features held fixed)...")
grid_search = GridSearchCV(
    tfidf_pipeline, param_grid, cv=5, scoring='f1_macro', n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)
print(f"\nBest CV Macro-F1 (TF-IDF only): {grid_search.best_score_:.4f}")
print(f"Winning TF-IDF/clf params: {grid_search.best_params_}")

best_tfidf_params = {
    k.replace('tfidf__', ''): v for k, v in grid_search.best_params_.items() if k.startswith('tfidf__')
}
best_C = grid_search.best_params_['clf__C']

final_tfidf = TfidfVectorizer(**best_tfidf_params).fit(X_train)
X_train_tfidf = final_tfidf.transform(X_train).toarray()
X_test_tfidf = final_tfidf.transform(X_test).toarray()

# ---------------------------------------------------------------------------
# Three conditions for a clean ablation: TF-IDF only, TF-IDF + exact-match,
# and TF-IDF + exact + semantic.
# ---------------------------------------------------------------------------
def train_and_eval(X_train_feats, X_test_feats, label):
    # Note the use of custom_weights here as well!
    model = LogisticRegression(
        C=best_C, class_weight=custom_weights, random_state=42, max_iter=2000
    ).fit(X_train_feats, y_train)

    y_pred = model.predict(X_test_feats)
    print(f"\n=======================================================")
    print(f"--- {label} ---")
    print(f"=======================================================")
    print(classification_report(y_test, y_pred))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred))
    return y_pred


X_train_exact = np.hstack([X_train_tfidf, X_train_lex])
X_test_exact = np.hstack([X_test_tfidf, X_test_lex])

# Run all 3 Ablations:
print("\n[Running Final Evaluations on Locked Test Set]")
train_and_eval(X_train_tfidf, X_test_tfidf, "TF-IDF ONLY (Baseline)")
train_and_eval(X_train_exact, X_test_exact, "TF-IDF + Exact-Match Lexicon")

X_train_full = np.hstack([X_train_tfidf, X_train_lex, X_train_sem])
X_test_full = np.hstack([X_test_tfidf, X_test_lex, X_test_sem])
train_and_eval(X_train_full, X_test_full, "TF-IDF + Exact-Match + Semantic Lexicon")

Loading data...
Precomputing exact-match lexicon features...
Precomputing semantic lexicon features (this loads a sentence-transformer model)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Starting Grid Search on TF-IDF + classifier (lexicon features held fixed)...
Fitting 5 folds for each of 36 candidates, totalling 180 fits

Best CV Macro-F1 (TF-IDF only): 0.7098
Winning TF-IDF/clf params: {'clf__C': 1.0, 'tfidf__max_features': 3000, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2)}

[Running Final Evaluations on Locked Test Set]

--- TF-IDF ONLY (Baseline) ---
              precision    recall  f1-score   support

           1       0.80      0.77      0.79       420
           2       0.69      0.73      0.71       308
           3       0.48      0.54      0.51       141
           4       0.81      0.66      0.73       111

    accuracy                           0.71       980
   macro avg       0.70      0.68      0.68       980
weighted avg       0.72      0.71      0.72       980

Confusion matrix:
[[325  53  37   5]
 [ 45 225  33   5]
 [ 20  38  76   7]
 [ 15  10  13  73]]

--- TF-IDF + Exact-Match Lexicon ---
              precision    recall  f1-score   supp

array([1, 2, 2, 2, 4, 1, 3, 2, 1, 1, 3, 1, 1, 1, 1, 1, 1, 3, 2, 2, 1, 3,
       2, 2, 2, 3, 3, 2, 1, 2, 4, 2, 2, 4, 2, 3, 3, 4, 1, 2, 2, 1, 4, 1,
       1, 3, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 2, 1, 1, 1, 2, 2, 1, 1, 1, 2,
       1, 3, 1, 1, 2, 2, 1, 2, 2, 1, 2, 3, 2, 2, 1, 1, 3, 1, 1, 4, 1, 1,
       1, 1, 4, 3, 1, 1, 1, 2, 3, 1, 1, 1, 1, 2, 1, 2, 1, 2, 3, 3, 1, 1,
       1, 1, 2, 2, 1, 4, 1, 3, 2, 2, 2, 1, 1, 1, 2, 2, 2, 1, 2, 1, 1, 2,
       1, 2, 3, 4, 1, 1, 1, 1, 4, 3, 1, 3, 1, 1, 1, 1, 4, 1, 1, 3, 1, 1,
       2, 1, 1, 1, 1, 2, 1, 4, 1, 3, 1, 1, 1, 1, 2, 1, 1, 1, 3, 3, 4, 1,
       2, 2, 3, 1, 2, 2, 4, 1, 2, 2, 2, 4, 2, 3, 1, 4, 2, 2, 2, 1, 2, 1,
       4, 1, 1, 1, 2, 1, 3, 1, 2, 3, 1, 1, 2, 2, 1, 2, 1, 2, 1, 4, 1, 1,
       2, 2, 1, 4, 2, 1, 2, 4, 1, 2, 1, 3, 3, 1, 1, 2, 4, 4, 2, 2, 3, 1,
       3, 3, 4, 3, 2, 4, 1, 1, 4, 2, 3, 2, 1, 1, 2, 2, 1, 1, 2, 1, 2, 3,
       1, 2, 1, 1, 3, 4, 2, 1, 1, 1, 1, 1, 2, 1, 2, 1, 1, 1, 2, 1, 2, 2,
       2, 3, 1, 4, 1, 4, 1, 1, 3, 2, 2, 4, 1, 1, 1,

In [ ]:
#block_11
import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix

from sentence_transformers import SentenceTransformer


# =============================================================================
# 1. LOAD DATA
# =============================================================================

print("Loading data...")

train_df = pd.read_csv("train_dataset.csv").dropna(
    subset=["posts", "labels"]
)

test_df = pd.read_csv("test_dataset.csv").dropna(
    subset=["posts", "labels"]
)

lexicon_df = pd.read_excel(
    "/content/Bangla_Depression_Lexicon_latest.xlsx"
).dropna(
    subset=["term_surface", "category_name"]
)


X_train = train_df["posts"].astype(str).values
y_train = train_df["labels"].astype(int).values

X_test = test_df["posts"].astype(str).values
y_test = test_df["labels"].astype(int).values


print(f"Train size: {len(X_train)}")
print(f"Test size : {len(X_test)}")
print(f"Classes   : {np.unique(y_train)}")


# =============================================================================
# 2. LEXICON SETUP
# =============================================================================

categories = sorted(
    lexicon_df["category_name"].unique()
)

cat_phrases = {
    cat: lexicon_df[
        lexicon_df["category_name"] == cat
    ]["term_surface"]
    .astype(str)
    .tolist()
    for cat in categories
}


has_severity_weight = (
    "severity_association" in lexicon_df.columns
)

if has_severity_weight:

    cat_weights = {
        cat: dict(
            zip(
                lexicon_df[
                    lexicon_df["category_name"] == cat
                ]["term_surface"].astype(str),

                lexicon_df[
                    lexicon_df["category_name"] == cat
                ]["severity_association"].astype(float)
            )
        )
        for cat in categories
    }


print("\nLexicon categories:")
for cat in categories:
    print(f"  {cat}: {len(cat_phrases[cat])} terms")


# =============================================================================
# 3. TEXT CLEANING
# =============================================================================

PUNCT_RE = re.compile(
    r'[।\.,\?!;:"\'\(\)\[\]\{\}/]'
)

SEGMENT_SPLIT_RE = re.compile(
    r'[।\n]|[০-৯0-9]+[/\.]'
)


def clean_text(text):

    return (
        " "
        + PUNCT_RE.sub(" ", text).strip()
        + " "
    )


def split_segments(text):

    segments = [
        s.strip()
        for s in SEGMENT_SPLIT_RE.split(text)
        if s.strip()
    ]

    return segments if segments else [text.strip()]


# =============================================================================
# 4. EXACT-MATCH LEXICON FEATURES
# =============================================================================

class LexiconFeatureExtractor(
    BaseEstimator,
    TransformerMixin
):

    def fit(self, X, y=None):

        raw = self._raw_features(X)

        self.scaler_ = MinMaxScaler().fit(raw)

        return self


    def transform(self, X):

        raw = self._raw_features(X)

        return self.scaler_.transform(raw)


    def _raw_features(self, X):

        features = []

        for text in X:

            text_clean = clean_text(text)

            word_count = max(
                len(text.split()),
                1
            )

            cat_counts = []
            cat_weighted = []

            for cat in categories:

                phrases = cat_phrases[cat]

                count = sum(
                    1
                    for phrase in phrases
                    if f" {phrase} " in text_clean
                )

                cat_counts.append(count)

                if has_severity_weight:

                    weighted = sum(
                        cat_weights[cat].get(
                            phrase,
                            0.0
                        )
                        for phrase in phrases
                        if f" {phrase} " in text_clean
                    )

                    cat_weighted.append(weighted)


            total = sum(cat_counts)

            # Number of categories represented
            breadth_score = sum(
                1
                for c in cat_counts
                if c > 0
            )

            # Proportion of matches belonging to each category
            category_proportions = [
                (c / total)
                if total > 0
                else 0.0
                for c in cat_counts
            ]

            # Normalize by document length
            category_density = [
                c / word_count
                for c in cat_counts
            ]


            row = []

            # Six category counts
            row += cat_counts

            # Total lexicon matches
            row += [total]

            # Number of distinct categories hit
            row += [breadth_score]

            # Category proportions
            row += category_proportions

            # Category density
            row += category_density

            # Optional severity association
            if has_severity_weight:
                row += cat_weighted

            features.append(row)


        return np.array(features)


# =============================================================================
# 5. SEMANTIC LEXICON FEATURES
# =============================================================================

class SemanticLexiconFeatureExtractor(
    BaseEstimator,
    TransformerMixin
):

    def __init__(
        self,
        cat_phrases,
        model_name="paraphrase-multilingual-mpnet-base-v2"
    ):

        self.cat_phrases = cat_phrases
        self.model_name = model_name


    def fit(self, X, y=None):

        print(
            f"Loading semantic model: {self.model_name}"
        )

        self.model_ = SentenceTransformer(
            self.model_name
        )

        self.categories_ = sorted(
            self.cat_phrases.keys()
        )

        self.term_embeds_ = {

            cat: self.model_.encode(
                self.cat_phrases[cat],
                normalize_embeddings=True,
                show_progress_bar=False
            )

            for cat in self.categories_
        }

        raw = self._raw_features(X)

        self.scaler_ = MinMaxScaler().fit(raw)

        return self


    def transform(self, X):

        raw = self._raw_features(X)

        return self.scaler_.transform(raw)


    def _raw_features(self, X):

        features = []

        for text in X:

            text_clean = clean_text(text)

            word_count = max(
                len(text.split()),
                1
            )


            # -------------------------------------------------------------
            # EXACT MATCH FEATURES
            # -------------------------------------------------------------

            cat_counts = []
            cat_weighted = []

            for cat in categories:

                phrases = cat_phrases[cat]

                count = sum(
                    1
                    for phrase in phrases
                    if f" {phrase} " in text_clean
                )

                cat_counts.append(count)

                if has_severity_weight:

                    weighted = sum(
                        cat_weights[cat].get(
                            phrase,
                            0.0
                        )
                        for phrase in phrases
                        if f" {phrase} " in text_clean
                    )

                    cat_weighted.append(weighted)


            total = sum(cat_counts)

            breadth_score = sum(
                1
                for c in cat_counts
                if c > 0
            )


            # -------------------------------------------------------------
            # SEMANTIC FEATURES
            # -------------------------------------------------------------

            segments = split_segments(text)

            segment_embeddings = self.model_.encode(
                segments,
                normalize_embeddings=True,
                show_progress_bar=False
            )


            semantic_scores = []

            for cat in self.categories_:

                term_embeddings = self.term_embeds_[cat]

                # cosine similarity because both are normalized
                similarities = np.matmul(
                    segment_embeddings,
                    term_embeddings.T
                )

                # Maximum semantic similarity in document
                max_similarity = (
                    similarities.max()
                    if similarities.size > 0
                    else 0.0
                )

                semantic_scores.append(
                    max_similarity
                )


            # -------------------------------------------------------------
            # OTHER FEATURES
            # -------------------------------------------------------------

            category_proportions = [
                (c / total)
                if total > 0
                else 0.0
                for c in cat_counts
            ]

            category_density = [
                c / word_count
                for c in cat_counts
            ]


            row = []

            # Exact lexical features
            row += cat_counts
            row += [total]
            row += [breadth_score]

            # Proportions
            row += category_proportions

            # Density
            row += category_density

            # Severity weights
            if has_severity_weight:
                row += cat_weighted

            # Semantic similarity for each category
            row += semantic_scores

            features.append(row)


        return np.array(features)


# =============================================================================
# 6. PRECOMPUTE LEXICON FEATURES
# =============================================================================

print(
    "\nPrecomputing exact-match lexicon features..."
)

lex_extractor = (
    LexiconFeatureExtractor()
    .fit(X_train)
)

X_train_lex = (
    lex_extractor.transform(X_train)
)

X_test_lex = (
    lex_extractor.transform(X_test)
)


# =============================================================================
# 7. PRECOMPUTE SEMANTIC FEATURES
# =============================================================================

print(
    "\nPrecomputing semantic lexicon features..."
)

sem_extractor = (
    SemanticLexiconFeatureExtractor(
        cat_phrases
    )
    .fit(X_train)
)

X_train_sem = (
    sem_extractor.transform(X_train)
)

X_test_sem = (
    sem_extractor.transform(X_test)
)


# =============================================================================
# 8. TF-IDF + LOGISTIC REGRESSION GRID SEARCH
# =============================================================================

print(
    "\n======================================================="
)
print("OPTIMIZING TF-IDF + LOGISTIC REGRESSION")
print("=======================================================")


cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


pipeline = Pipeline([

    (
        "tfidf",
        TfidfVectorizer(
            sublinear_tf=True
        )
    ),

    (
        "clf",
        LogisticRegression(
            max_iter=3000,
            random_state=42
        )
    )
])


param_grid = {

    # More vocabulary capacity
    "tfidf__max_features": [
        5000,
        10000,
        20000
    ],

    # Include trigrams
    "tfidf__ngram_range": [
        (1, 2),
        (1, 3)
    ],

    # Remove extremely rare terms
    "tfidf__min_df": [
        2,
        3,
        5
    ],

    # Try a wider regularization range
    "clf__C": [
        0.1,
        0.3,
        1.0,
        3.0,
        10.0
    ],

    # Compare balanced vs no weighting
    "clf__class_weight": [
        None,
        "balanced"
    ]
}


grid_search = GridSearchCV(

    pipeline,

    param_grid,

    cv=cv,

    scoring="f1_macro",

    n_jobs=-1,

    verbose=2,

    refit=True
)


grid_search.fit(
    X_train,
    y_train
)


print(
    "\nBest CV Macro-F1:",
    round(
        grid_search.best_score_,
        4
    )
)

print(
    "\nBest parameters:"
)

for key, value in grid_search.best_params_.items():

    print(
        f"  {key}: {value}"
    )


# =============================================================================
# 9. FIT FINAL TF-IDF USING BEST PARAMETERS
# =============================================================================

best_params = grid_search.best_params_


best_tfidf_params = {

    key.replace(
        "tfidf__",
        ""
    ): value

    for key, value
    in best_params.items()

    if key.startswith("tfidf__")
}


best_C = best_params["clf__C"]

best_class_weight = (
    best_params["clf__class_weight"]
)


print(
    "\nSelected:"
)

print(
    "TF-IDF:",
    best_tfidf_params
)

print(
    "C:",
    best_C
)

print(
    "Class weight:",
    best_class_weight
)


final_tfidf = (
    TfidfVectorizer(
        **best_tfidf_params
    )
    .fit(X_train)
)


X_train_tfidf = (
    final_tfidf
    .transform(X_train)
    .toarray()
)

X_test_tfidf = (
    final_tfidf
    .transform(X_test)
    .toarray()
)


# =============================================================================
# 10. FINAL MODEL FUNCTION
# =============================================================================

def train_and_eval(
    X_train_feats,
    X_test_feats,
    label
):

    print(
        "\n======================================================="
    )

    print(
        f"--- {label} ---"
    )

    print(
        "======================================================="
    )


    model = LogisticRegression(

        C=best_C,

        class_weight=best_class_weight,

        random_state=42,

        max_iter=3000
    )


    model.fit(
        X_train_feats,
        y_train
    )


    y_pred = model.predict(
        X_test_feats
    )


    print(
        classification_report(
            y_test,
            y_pred,
            digits=4
        )
    )


    print(
        "Confusion matrix:"
    )

    print(
        confusion_matrix(
            y_test,
            y_pred
        )
    )


    return model, y_pred


# =============================================================================
# 11. BASELINE
# =============================================================================

model_baseline, pred_baseline = train_and_eval(

    X_train_tfidf,

    X_test_tfidf,

    "TF-IDF ONLY (BASELINE)"
)


# =============================================================================
# 12. TF-IDF + EXACT LEXICON
# =============================================================================

X_train_exact = np.hstack([
    X_train_tfidf,
    X_train_lex
])

X_test_exact = np.hstack([
    X_test_tfidf,
    X_test_lex
])


model_exact, pred_exact = train_and_eval(

    X_train_exact,

    X_test_exact,

    "TF-IDF + EXACT-MATCH LEXICON"
)


# =============================================================================
# 13. TF-IDF + EXACT + SEMANTIC LEXICON
# =============================================================================

X_train_full = np.hstack([
    X_train_tfidf,
    X_train_lex,
    X_train_sem
])

X_test_full = np.hstack([
    X_test_tfidf,
    X_test_lex,
    X_test_sem
])


model_full, pred_full = train_and_eval(

    X_train_full,

    X_test_full,

    "TF-IDF + EXACT + SEMANTIC LEXICON"
)


print(
    "\n======================================================="
)

print(
    "FINAL EVALUATION COMPLETE"
)

print(
    "======================================================="
)

Loading data...
Train size: 3917
Test size : 980
Classes   : [1 2 3 4]

Lexicon categories:
  Anhedonia & Social Withdrawal: 51 terms
  Cognitive Impairment & Concentration: 40 terms
  Crisis, Death & Psychache: 74 terms
  Guilt, Failure & Worthlessness: 46 terms
  Psychiatric Distress & Depressed Affect: 107 terms
  Somatic Distress & Physical Complaints: 101 terms

Precomputing exact-match lexicon features...

Precomputing semantic lexicon features...
Loading semantic model: paraphrase-multilingual-mpnet-base-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


OPTIMIZING TF-IDF + LOGISTIC REGRESSION
Fitting 5 folds for each of 180 candidates, totalling 900 fits

Best CV Macro-F1: 0.6972

Best parameters:
  clf__C: 10.0
  clf__class_weight: balanced
  tfidf__max_features: 10000
  tfidf__min_df: 2
  tfidf__ngram_range: (1, 2)

Selected:
TF-IDF: {'max_features': 10000, 'min_df': 2, 'ngram_range': (1, 2)}
C: 10.0
Class weight: balanced

--- TF-IDF ONLY (BASELINE) ---
              precision    recall  f1-score   support

           1     0.7654    0.8310    0.7968       420
           2     0.7235    0.6883    0.7055       308
           3     0.5308    0.4894    0.5092       141
           4     0.7624    0.6937    0.7264       111

    accuracy                         0.7214       980
   macro avg     0.6955    0.6756    0.6845       980
weighted avg     0.7181    0.7214    0.7188       980

Confusion matrix:
[[349  36  26   9]
 [ 64 212  26   6]
 [ 27  36  69   9]
 [ 16   9   9  77]]

--- TF-IDF + EXACT-MATCH LEXICON ---
              precis

In [ ]:
pip install sentence_transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 22.2 MB/s eta 0:00:00


In [ ]:
pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 8.3 MB/s eta 0:00:00


In [ ]:
#block_12
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from statsmodels.stats.contingency_tables import mcnemar
from sentence_transformers import SentenceTransformer

# ===========================================================================
# 1. LOAD DATA
# ===========================================================================
print("Loading data...")
train_df = pd.read_csv("train_dataset.csv").dropna(subset=['posts', 'labels'])
test_df = pd.read_csv("test_dataset.csv").dropna(subset=['posts', 'labels'])
lexicon_df = pd.read_excel("/content/Bangla_Depression_Lexicon_latest.xlsx").dropna(
    subset=['term_surface', 'category_name']
)

X_train = train_df['posts'].astype(str).values
y_train = train_df['labels'].astype(int).values
X_test = test_df['posts'].astype(str).values
y_test = test_df['labels'].astype(int).values

categories = sorted(lexicon_df['category_name'].unique())
cat_phrases = {
    cat: lexicon_df[lexicon_df['category_name'] == cat]['term_surface'].astype(str).tolist()
    for cat in categories
}

has_severity_weight = 'severity_association' in lexicon_df.columns
if has_severity_weight:
    cat_weights = {
        cat: dict(zip(
            lexicon_df[lexicon_df['category_name'] == cat]['term_surface'].astype(str),
            lexicon_df[lexicon_df['category_name'] == cat]['severity_association'].astype(float)
        ))
        for cat in categories
    }

PUNCT_RE = re.compile(r'[।\.,\?!;:"\'\(\)\[\]\{\}/]')
SEGMENT_SPLIT_RE = re.compile(r'[।\n]|[০-৯0-9]+[/\.]')


def clean_text(text):
    return " " + PUNCT_RE.sub(' ', text).strip() + " "


def split_segments(text):
    segments = [s.strip() for s in SEGMENT_SPLIT_RE.split(text) if s.strip()]
    return segments if segments else [text.strip()]


# ===========================================================================
# 2. EXACT-MATCH LEXICON FEATURE EXTRACTOR
# ===========================================================================
class LexiconFeatureExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        raw = self._raw_features(X)
        self.scaler_ = MinMaxScaler().fit(raw)
        return self

    def transform(self, X):
        raw = self._raw_features(X)
        return self.scaler_.transform(raw)

    def _raw_features(self, X):
        features = []
        for text in X:
            text_clean = clean_text(text)
            word_count = max(len(text.split()), 1)
            cat_counts, cat_weighted = [], []
            for cat in categories:
                phrases = cat_phrases[cat]
                count = sum(1 for phrase in phrases if f" {phrase} " in text_clean)
                cat_counts.append(count)
                if has_severity_weight:
                    weighted = sum(
                        cat_weights[cat].get(phrase, 0.0)
                        for phrase in phrases
                        if f" {phrase} " in text_clean
                    )
                    cat_weighted.append(weighted)
            total = sum(cat_counts)
            row = []
            row += cat_counts
            row += [total]
            row += [(c / total) if total > 0 else 0.0 for c in cat_counts]
            row += [c / word_count for c in cat_counts]
            if has_severity_weight:
                row += cat_weighted
            features.append(row)
        return np.array(features)


# ===========================================================================
# 3. SEMANTIC LEXICON FEATURE EXTRACTOR
# ===========================================================================
class SemanticLexiconFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, cat_phrases, model_name='paraphrase-multilingual-mpnet-base-v2'):
        self.cat_phrases = cat_phrases
        self.model_name = model_name

    def fit(self, X, y=None):
        self.model_ = SentenceTransformer(self.model_name)
        self.categories_ = sorted(self.cat_phrases.keys())
        self.term_embeds_ = {
            cat: self.model_.encode(self.cat_phrases[cat], normalize_embeddings=True)
            for cat in self.categories_
        }
        raw = self._raw_features(X)
        self.scaler_ = MinMaxScaler().fit(raw)
        return self

    def transform(self, X):
        raw = self._raw_features(X)
        return self.scaler_.transform(raw)

    def _raw_features(self, X):
        features = []
        for text in X:
            segments = split_segments(str(text))
            seg_embeds = self.model_.encode(segments, normalize_embeddings=True)
            row = []
            for cat in self.categories_:
                sims = seg_embeds @ self.term_embeds_[cat].T
                max_sim = sims.max() if sims.size else 0.0
                top3_mean = np.sort(sims.flatten())[-3:].mean() if sims.size else 0.0
                row += [max_sim, top3_mean]
            features.append(row)
        return np.array(features)


# ===========================================================================
# 4. PRECOMPUTE LEXICON FEATURES (once, outside grid search)
# ===========================================================================
print("Precomputing exact-match lexicon features...")
lex_extractor = LexiconFeatureExtractor().fit(X_train)
X_train_lex = lex_extractor.transform(X_train)
X_test_lex = lex_extractor.transform(X_test)

print("Precomputing semantic lexicon features (loads a sentence-transformer model)...")
sem_extractor = SemanticLexiconFeatureExtractor(cat_phrases).fit(X_train)
X_train_sem = sem_extractor.transform(X_train)
X_test_sem = sem_extractor.transform(X_test)

X_train_lexonly = np.hstack([X_train_lex, X_train_sem])
X_test_lexonly = np.hstack([X_test_lex, X_test_sem])

# ===========================================================================
# 5. GRID SEARCH: TF-IDF + LOGISTIC REGRESSION (text signal only)
# ===========================================================================
tfidf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(class_weight='balanced', random_state=42, max_iter=2000))
])

param_grid = {
    'tfidf__max_features': [3000, 5000, 10000],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df': [2, 5],
    'clf__C': [0.1, 1.0, 10.0]
}

print("Starting Grid Search on TF-IDF + classifier...")
grid_search = GridSearchCV(
    tfidf_pipeline, param_grid, cv=5, scoring='f1_macro', n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)
print(f"\nBest CV Macro-F1 (TF-IDF only): {grid_search.best_score_:.4f}")
print(f"Winning TF-IDF/clf params: {grid_search.best_params_}")

best_tfidf_params = {
    k.replace('tfidf__', ''): v for k, v in grid_search.best_params_.items() if k.startswith('tfidf__')
}
best_C = grid_search.best_params_['clf__C']

final_tfidf = TfidfVectorizer(**best_tfidf_params).fit(X_train)
X_train_tfidf = final_tfidf.transform(X_train).toarray()
X_test_tfidf = final_tfidf.transform(X_test).toarray()

# ===========================================================================
# 6. TRAIN + EVALUATE ALL CONDITIONS
# ===========================================================================
def train_and_eval(X_tr, X_te, label, C=best_C):
    model = LogisticRegression(
        C=C, class_weight='balanced', random_state=42, max_iter=2000
    ).fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    print(f"\n--- {label} ---")
    print(classification_report(y_test, y_pred))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred))
    print(f"Macro-F1: {f1_score(y_test, y_pred, average='macro'):.4f}")
    return y_pred


# Condition 1: TF-IDF only (baseline)
y_pred_baseline = train_and_eval(X_train_tfidf, X_test_tfidf, "TF-IDF Only (Baseline)")

# Condition 2: TF-IDF + Exact-Match Lexicon
X_train_exact = np.hstack([X_train_tfidf, X_train_lex])
X_test_exact = np.hstack([X_test_tfidf, X_test_lex])
y_pred_exact = train_and_eval(X_train_exact, X_test_exact, "TF-IDF + Exact-Match Lexicon")

# Condition 3: TF-IDF + Exact-Match + Semantic Lexicon
X_train_full = np.hstack([X_train_tfidf, X_train_lex, X_train_sem])
X_test_full = np.hstack([X_test_tfidf, X_test_lex, X_test_sem])
y_pred_full = train_and_eval(X_train_full, X_test_full, "TF-IDF + Exact + Semantic Lexicon")

# Condition 4: Lexicon only (no TF-IDF at all) — separately tuned C,
# since this feature space is far lower-dimensional than TF-IDF and
# reusing the TF-IDF-tuned C isn't a fair comparison.
lex_param_grid = {'C': [0.01, 0.1, 1.0, 10.0, 100.0]}
lex_grid = GridSearchCV(
    LogisticRegression(class_weight='balanced', random_state=42, max_iter=2000),
    lex_param_grid, cv=5, scoring='f1_macro', n_jobs=-1
)
lex_grid.fit(X_train_lexonly, y_train)
print(f"\nBest CV Macro-F1 (Lexicon only): {lex_grid.best_score_:.4f}, Best C: {lex_grid.best_params_}")
y_pred_lexonly = train_and_eval(X_train_lexonly, X_test_lexonly, "Lexicon Only (No TF-IDF)",
                                  C=lex_grid.best_params_['C'])

# ===========================================================================
# 7. McNEMAR'S TEST: is the lexicon-only error pattern significantly
# different from the TF-IDF-only baseline, or statistically indistinguishable
# ===========================================================================
def run_mcnemar(pred_a, pred_b, label_a, label_b):
    correct_a = (pred_a == y_test)
    correct_b = (pred_b == y_test)
    table = [
        [np.sum(correct_a & correct_b), np.sum(correct_a & ~correct_b)],
        [np.sum(~correct_a & correct_b), np.sum(~correct_a & ~correct_b)]
    ]
    result = mcnemar(table, exact=True)
    print(f"\nMcNemar's test — {label_a} vs {label_b}")
    print(f"Statistic: {result.statistic}, p-value: {result.pvalue:.4f}")
    return result


run_mcnemar(y_pred_baseline, y_pred_lexonly, "TF-IDF Only", "Lexicon Only")
run_mcnemar(y_pred_baseline, y_pred_full, "TF-IDF Only", "TF-IDF + Exact + Semantic")

# ===========================================================================
# 8. SUMMARY TABLE
# ===========================================================================
print("\n=== SUMMARY ===")
print(f"TF-IDF only:                    Macro-F1 = {f1_score(y_test, y_pred_baseline, average='macro'):.4f}")
print(f"TF-IDF + Exact Lexicon:         Macro-F1 = {f1_score(y_test, y_pred_exact, average='macro'):.4f}")
print(f"TF-IDF + Exact + Semantic:      Macro-F1 = {f1_score(y_test, y_pred_full, average='macro'):.4f}")
print(f"Lexicon only (no TF-IDF):       Macro-F1 = {f1_score(y_test, y_pred_lexonly, average='macro'):.4f}")

Loading data...
Precomputing exact-match lexicon features...
Precomputing semantic lexicon features (loads a sentence-transformer model)...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

KeyboardInterrupt: 

In [ ]:
pip install statsmodels


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 133.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.3/233.3 kB 23.5 MB/s eta 0:00:00


In [ ]:
lex_extractor = LexiconFeatureExtractor().fit(X_train)
X_train_lex = lex_extractor.transform(X_train)
X_test_lex = lex_extractor.transform(X_test)

sem_extractor = SemanticLexiconFeatureExtractor(cat_phrases).fit(X_train)
X_train_sem = sem_extractor.transform(X_train)
X_test_sem = sem_extractor.transform(X_test)

NameError: name 'LexiconFeatureExtractor' is not defined

In [ ]:
#block_13
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from statsmodels.stats.contingency_tables import mcnemar
from sentence_transformers import SentenceTransformer

# ===========================================================================
# 1. LOAD DATA
# ===========================================================================
print("Loading data...")
train_df = pd.read_csv("train_dataset.csv").dropna(subset=['posts', 'labels'])
test_df = pd.read_csv("test_dataset.csv").dropna(subset=['posts', 'labels'])
lexicon_df = pd.read_excel("/content/Bangla_Depression_Lexicon_latest.xlsx").dropna(
    subset=['term_surface', 'category_name']
)

X_train = train_df['posts'].astype(str).values
y_train = train_df['labels'].astype(int).values
X_test = test_df['posts'].astype(str).values
y_test = test_df['labels'].astype(int).values

categories = sorted(lexicon_df['category_name'].unique())
cat_phrases = {
    cat: lexicon_df[lexicon_df['category_name'] == cat]['term_surface'].astype(str).tolist()
    for cat in categories
}

has_severity_weight = 'severity_association' in lexicon_df.columns
if has_severity_weight:
    cat_weights = {
        cat: dict(zip(
            lexicon_df[lexicon_df['category_name'] == cat]['term_surface'].astype(str),
            lexicon_df[lexicon_df['category_name'] == cat]['severity_association'].astype(float)
        ))
        for cat in categories
    }

PUNCT_RE = re.compile(r'[।\.,\?!;:"\'\(\)\[\]\{\}/]')
SEGMENT_SPLIT_RE = re.compile(r'[।\n]|[০-৯0-9]+[/\.]')


def clean_text(text):
    return " " + PUNCT_RE.sub(' ', text).strip() + " "


def split_segments(text):
    segments = [s.strip() for s in SEGMENT_SPLIT_RE.split(text) if s.strip()]
    return segments if segments else [text.strip()]


# ===========================================================================
# 2. EXACT-MATCH LEXICON FEATURE EXTRACTOR
# ===========================================================================
class LexiconFeatureExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        raw = self._raw_features(X)
        self.scaler_ = MinMaxScaler().fit(raw)
        return self

    def transform(self, X):
        raw = self._raw_features(X)
        return self.scaler_.transform(raw)

    def _raw_features(self, X):
        features = []
        for text in X:
            text_clean = clean_text(text)
            word_count = max(len(text.split()), 1)
            cat_counts, cat_weighted = [], []
            for cat in categories:
                phrases = cat_phrases[cat]
                count = sum(1 for phrase in phrases if f" {phrase} " in text_clean)
                cat_counts.append(count)
                if has_severity_weight:
                    weighted = sum(
                        cat_weights[cat].get(phrase, 0.0)
                        for phrase in phrases
                        if f" {phrase} " in text_clean
                    )
                    cat_weighted.append(weighted)
            total = sum(cat_counts)
            row = []
            row += cat_counts
            row += [total]
            row += [(c / total) if total > 0 else 0.0 for c in cat_counts]
            row += [c / word_count for c in cat_counts]
            if has_severity_weight:
                row += cat_weighted
            features.append(row)
        return np.array(features)


# ===========================================================================
# 3. SEMANTIC LEXICON FEATURE EXTRACTOR
# ===========================================================================
class SemanticLexiconFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, cat_phrases, model_name='paraphrase-multilingual-mpnet-base-v2'):
        self.cat_phrases = cat_phrases
        self.model_name = model_name

    def fit(self, X, y=None):
        self.model_ = SentenceTransformer(self.model_name)
        self.categories_ = sorted(self.cat_phrases.keys())
        self.term_embeds_ = {
            cat: self.model_.encode(self.cat_phrases[cat], normalize_embeddings=True)
            for cat in self.categories_
        }
        raw = self._raw_features(X)
        self.scaler_ = MinMaxScaler().fit(raw)
        return self

    def transform(self, X):
        raw = self._raw_features(X)
        return self.scaler_.transform(raw)

    def _raw_features(self, X):
        features = []
        for text in X:
            segments = split_segments(str(text))
            seg_embeds = self.model_.encode(segments, normalize_embeddings=True)
            row = []
            for cat in self.categories_:
                sims = seg_embeds @ self.term_embeds_[cat].T
                max_sim = sims.max() if sims.size else 0.0
                top3_mean = np.sort(sims.flatten())[-3:].mean() if sims.size else 0.0
                row += [max_sim, top3_mean]
            features.append(row)
        return np.array(features)


# ===========================================================================
# 4. PRECOMPUTE LEXICON FEATURES (once, outside grid search)
# ===========================================================================
print("Precomputing exact-match lexicon features...")
lex_extractor = LexiconFeatureExtractor().fit(X_train)
X_train_lex = lex_extractor.transform(X_train)
X_test_lex = lex_extractor.transform(X_test)

print("Precomputing semantic lexicon features (loads a sentence-transformer model)...")
sem_extractor = SemanticLexiconFeatureExtractor(cat_phrases).fit(X_train)
X_train_sem = sem_extractor.transform(X_train)
X_test_sem = sem_extractor.transform(X_test)

X_train_lexonly = np.hstack([X_train_lex, X_train_sem])
X_test_lexonly = np.hstack([X_test_lex, X_test_sem])

# ===========================================================================
# 5. GRID SEARCH: TF-IDF + LOGISTIC REGRESSION (text signal only)
# ===========================================================================
tfidf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(class_weight='balanced', random_state=42, max_iter=2000))
])

param_grid = {
    'tfidf__max_features': [3000, 5000, 10000],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df': [2, 5],
    'clf__C': [0.1, 1.0, 10.0]
}

print("Starting Grid Search on TF-IDF + classifier...")
grid_search = GridSearchCV(
    tfidf_pipeline, param_grid, cv=5, scoring='f1_macro', n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)
print(f"\nBest CV Macro-F1 (TF-IDF only): {grid_search.best_score_:.4f}")
print(f"Winning TF-IDF/clf params: {grid_search.best_params_}")

best_tfidf_params = {
    k.replace('tfidf__', ''): v for k, v in grid_search.best_params_.items() if k.startswith('tfidf__')
}
best_C = grid_search.best_params_['clf__C']

final_tfidf = TfidfVectorizer(**best_tfidf_params).fit(X_train)
X_train_tfidf = final_tfidf.transform(X_train).toarray()
X_test_tfidf = final_tfidf.transform(X_test).toarray()

# ===========================================================================
# 6. TRAIN + EVALUATE ALL CONDITIONS
# ===========================================================================
def train_and_eval(X_train_feats, X_test_feats, label, apply_override=True):
    # Train the model with our custom weights!
    model = LogisticRegression(
        C=best_C, class_weight=custom_weights, random_state=42, max_iter=2000
    ).fit(X_train_feats, y_train)

    # Get the ML predictions
    y_pred = model.predict(X_test_feats)

    # ==========================================
    # CLINICAL OVERRIDE (The 7-Post Rescue!)
    # ==========================================
    if apply_override:
        # NOTE: Change this string to match your exact Excel category name!
        crisis_category = "Crisis, Death & Psychache"

        # Get all the deadly phrases from the lexicon
        if crisis_category in cat_phrases:
            crisis_phrases = cat_phrases[crisis_category]
            overrides_applied = 0

            for i, text in enumerate(X_test):
                text_clean = clean_text(text)
                # If the post contains ANY suicide/crisis phrase
                if any(f" {phrase} " in text_clean for phrase in crisis_phrases):
                    # And the model foolishly predicted 1, 2, or 3
                    if y_pred[i] < 4:
                        y_pred[i] = 4  # Force upgrade to Severe!
                        overrides_applied += 1

            print(f"\n🚨 [CLINICAL OVERRIDE] Rescued {overrides_applied} severe patients!")
        else:
            print(f"\n⚠️ WARNING: Could not find category '{crisis_category}'. Check spelling!")

    print(f"\n=======================================================")
    print(f"--- {label} ---")
    print(f"=======================================================")
    print(classification_report(y_test, y_pred))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred))
    return y_pred


# Condition 1: TF-IDF only (baseline)
y_pred_baseline = train_and_eval(X_train_tfidf, X_test_tfidf, "TF-IDF Only (Baseline)")

# Condition 2: TF-IDF + Exact-Match Lexicon
X_train_exact = np.hstack([X_train_tfidf, X_train_lex])
X_test_exact = np.hstack([X_test_tfidf, X_test_lex])
y_pred_exact = train_and_eval(X_train_exact, X_test_exact, "TF-IDF + Exact-Match Lexicon")

# Condition 3: TF-IDF + Exact-Match + Semantic Lexicon
X_train_full = np.hstack([X_train_tfidf, X_train_lex, X_train_sem])
X_test_full = np.hstack([X_test_tfidf, X_test_lex, X_test_sem])
y_pred_full = train_and_eval(X_train_full, X_test_full, "TF-IDF + Exact + Semantic Lexicon")

# Condition 4: Lexicon only (no TF-IDF at all) — separately tuned C,
# since this feature space is far lower-dimensional than TF-IDF and
# reusing the TF-IDF-tuned C isn't a fair comparison.
lex_param_grid = {'C': [0.01, 0.1, 1.0, 10.0, 100.0]}
lex_grid = GridSearchCV(
    LogisticRegression(class_weight='balanced', random_state=42, max_iter=2000),
    lex_param_grid, cv=5, scoring='f1_macro', n_jobs=-1
)
lex_grid.fit(X_train_lexonly, y_train)
print(f"\nBest CV Macro-F1 (Lexicon only): {lex_grid.best_score_:.4f}, Best C: {lex_grid.best_params_}")
y_pred_lexonly = train_and_eval(X_train_lexonly, X_test_lexonly, "Lexicon Only (No TF-IDF)",
                                  C=lex_grid.best_params_['C'])

# ===========================================================================
# 7. McNEMAR'S TEST: is the lexicon-only error pattern significantly
# different from the TF-IDF-only baseline, or statistically indistinguishable
# ===========================================================================
def run_mcnemar(pred_a, pred_b, label_a, label_b):
    correct_a = (pred_a == y_test)
    correct_b = (pred_b == y_test)
    table = [
        [np.sum(correct_a & correct_b), np.sum(correct_a & ~correct_b)],
        [np.sum(~correct_a & correct_b), np.sum(~correct_a & ~correct_b)]
    ]
    result = mcnemar(table, exact=True)
    print(f"\nMcNemar's test — {label_a} vs {label_b}")
    print(f"Statistic: {result.statistic}, p-value: {result.pvalue:.4f}")
    return result


run_mcnemar(y_pred_baseline, y_pred_lexonly, "TF-IDF Only", "Lexicon Only")
run_mcnemar(y_pred_baseline, y_pred_full, "TF-IDF Only", "TF-IDF + Exact + Semantic")

# ===========================================================================
# 8. SUMMARY TABLE
# ===========================================================================
print("\n=== SUMMARY ===")
print(f"TF-IDF only:                    Macro-F1 = {f1_score(y_test, y_pred_baseline, average='macro'):.4f}")
print(f"TF-IDF + Exact Lexicon:         Macro-F1 = {f1_score(y_test, y_pred_exact, average='macro'):.4f}")
print(f"TF-IDF + Exact + Semantic:      Macro-F1 = {f1_score(y_test, y_pred_full, average='macro'):.4f}")
print(f"Lexicon only (no TF-IDF):       Macro-F1 = {f1_score(y_test, y_pred_lexonly, average='macro'):.4f}")

Loading data...
Precomputing exact-match lexicon features...
Precomputing semantic lexicon features (loads a sentence-transformer model)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Starting Grid Search on TF-IDF + classifier...
Fitting 5 folds for each of 36 candidates, totalling 180 fits

Best CV Macro-F1 (TF-IDF only): 0.6903
Winning TF-IDF/clf params: {'clf__C': 10.0, 'tfidf__max_features': 10000, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2)}

🚨 [CLINICAL OVERRIDE] Rescued 42 severe patients!

--- TF-IDF Only (Baseline) ---
              precision    recall  f1-score   support

           1       0.78      0.80      0.79       420
           2       0.70      0.66      0.68       308
           3       0.51      0.45      0.48       141
           4       0.62      0.74      0.67       111

    accuracy                           0.70       980
   macro avg       0.65      0.66      0.65       980
weighted avg       0.70      0.70      0.70       980

Confusion matrix:
[[336  43  28  13]
 [ 60 204  24  20]
 [ 24  37  63  17]
 [ 13   8   8  82]]

🚨 [CLINICAL OVERRIDE] Rescued 32 severe patients!

--- TF-IDF + Exact-Match Lexicon ---
              precision  

TypeError: train_and_eval() got an unexpected keyword argument 'C'

In [ ]:
#block_14
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings('ignore')

print("Loading data...")
train_df = pd.read_csv("train_dataset.csv").dropna(subset=['posts', 'labels'])
test_df = pd.read_csv("test_dataset.csv").dropna(subset=['posts', 'labels'])
lexicon_df = pd.read_excel("/content/Bangla_Depression_Lexicon_latest.xlsx").dropna(
    subset=['term_surface', 'category_name']
)

X_train = train_df['posts'].astype(str).values
y_train = train_df['labels'].astype(int).values
X_test = test_df['posts'].astype(str).values
y_test = test_df['labels'].astype(int).values

# ---------------------------------------------------------------------------
# CUSTOM CLASS WEIGHTS: Forcing the SVM to respect Class 3
# ---------------------------------------------------------------------------
# Documented rationale: base weights follow inverse sqrt of class support
# (support: 1=420, 2=308, 3=141, 4=111 in the held-out test set; train
# proportions assumed similar), then class 3 is boosted further given it is
# the specific diagnosed failure point (mild<->moderate boundary confusion
# in the confusion matrix). These are treated as a starting point, not a
# final answer -- see CLASS_WEIGHT_GRID below, which searches around them.
custom_weights = {1: 1.0, 2: 2.0, 3: 4.0, 4: 2.5}

# A small grid of alternative weight sets to compare against the hand-picked
# one and against sklearn's 'balanced' option. Keeps the choice defensible
# rather than arbitrary -- report whichever wins on macro-F1 / class-3 F1.
CLASS_WEIGHT_GRID = {
    'hand_picked': custom_weights,
    'balanced': 'balanced',
    'moderate_boost_only': {1: 1.0, 2: 1.0, 3: 3.0, 4: 1.0},
    'inverse_sqrt_support': {1: 1.0, 2: 1.17, 3: 1.73, 4: 1.95},  # sqrt(420/support)
}

categories = sorted(lexicon_df['category_name'].unique())
cat_phrases = {
    cat: lexicon_df[lexicon_df['category_name'] == cat]['term_surface'].astype(str).tolist()
    for cat in categories
}
has_severity_weight = 'severity_association' in lexicon_df.columns
if has_severity_weight:
    cat_weights = {
        cat: dict(zip(
            lexicon_df[lexicon_df['category_name'] == cat]['term_surface'].astype(str),
            lexicon_df[lexicon_df['category_name'] == cat]['severity_association'].astype(float)
        ))
        for cat in categories
    }

PUNCT_RE = re.compile(r'[।\.,\?!;:"\'\(\)\[\]\{\}/]')
SEGMENT_SPLIT_RE = re.compile(r'[।\n]|[০-৯0-9]+[/\.]')

def clean_text(text):
    return " " + PUNCT_RE.sub(' ', text).strip() + " "

def split_segments(text):
    segments = [s.strip() for s in SEGMENT_SPLIT_RE.split(text) if s.strip()]
    return segments if segments else [text.strip()]

# ---------------------------------------------------------------------------
# Block 1: Exact-Match Lexicon Features (Counts, Ratios, Breadth, Zero-Flag)
# ---------------------------------------------------------------------------
class LexiconFeatureExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        raw = self._raw_features(X)
        self.scaler_ = MinMaxScaler().fit(raw)
        return self

    def transform(self, X):
        raw = self._raw_features(X)
        return self.scaler_.transform(raw)

    def _raw_features(self, X):
        features = []
        for text in X:
            text_clean = clean_text(text)
            word_count = max(len(text.split()), 1)
            cat_counts, cat_weighted = [], []
            for cat in categories:
                phrases = cat_phrases[cat]
                count = sum(1 for phrase in phrases if f" {phrase} " in text_clean)
                cat_counts.append(count)
                if has_severity_weight:
                    weighted = sum(
                        cat_weights[cat].get(phrase, 0.0)
                        for phrase in phrases
                        if f" {phrase} " in text_clean
                    )
                    cat_weighted.append(weighted)
            total = sum(cat_counts)

            # Clinical Features
            breadth_score = sum(1 for c in cat_counts if c > 0)
            is_symptom_free = 1.0 if total == 0 else 0.0

            row = []
            row += cat_counts
            row += [total]
            row += [breadth_score]
            row += [is_symptom_free]
            row += [(c / total) if total > 0 else 0.0 for c in cat_counts]
            row += [c / word_count for c in cat_counts]
            if has_severity_weight:
                row += cat_weighted
            features.append(row)
        return np.array(features)

# ---------------------------------------------------------------------------
# Block 2: Semantic Lexicon Features
# ---------------------------------------------------------------------------
class SemanticLexiconFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, cat_phrases, model_name='paraphrase-multilingual-mpnet-base-v2'):
        self.cat_phrases = cat_phrases
        self.model_name = model_name

    def fit(self, X, y=None):
        self.model_ = SentenceTransformer(self.model_name)
        self.categories_ = sorted(self.cat_phrases.keys())
        self.term_embeds_ = {
            cat: self.model_.encode(self.cat_phrases[cat], normalize_embeddings=True)
            for cat in self.categories_
        }
        raw = self._raw_features(X)
        self.scaler_ = MinMaxScaler().fit(raw)
        return self

    def transform(self, X):
        raw = self._raw_features(X)
        return self.scaler_.transform(raw)

    def _raw_features(self, X):
        features = []
        for text in X:
            segments = split_segments(str(text))
            seg_embeds = self.model_.encode(segments, normalize_embeddings=True)
            row = []
            for cat in self.categories_:
                sims = seg_embeds @ self.term_embeds_[cat].T
                max_sim = sims.max() if sims.size else 0.0
                top3_mean = np.sort(sims.flatten())[-3:].mean() if sims.size else 0.0
                row += [max_sim, top3_mean]
            features.append(row)
        return np.array(features)

# ---------------------------------------------------------------------------
# Precompute features (once, reused across all evaluations)
# ---------------------------------------------------------------------------
print("Precomputing exact-match lexicon features...")
lex_extractor = LexiconFeatureExtractor().fit(X_train)
X_train_lex = lex_extractor.transform(X_train)
X_test_lex = lex_extractor.transform(X_test)

print("Precomputing semantic lexicon features (this loads a sentence-transformer model)...")
sem_extractor = SemanticLexiconFeatureExtractor(cat_phrases).fit(X_train)
X_train_sem = sem_extractor.transform(X_train)
X_test_sem = sem_extractor.transform(X_test)

# ---------------------------------------------------------------------------
# FIX #1: Sanity-check feature scale compatibility before concatenating
# ---------------------------------------------------------------------------
def report_scale(name, arr):
    print(f"  {name}: min={arr.min():.4f}  max={arr.max():.4f}  mean={arr.mean():.4f}")

print("\n[Feature scale sanity check]")
report_scale("Lexicon (exact)", X_train_lex)
report_scale("Semantic", X_train_sem)
# TF-IDF is checked after it's built below.

# ---------------------------------------------------------------------------
# TF-IDF grid search (structural hyperparameters only -- these are re-tuned
# for C, per feature set, below in FIX #2)
# ---------------------------------------------------------------------------
tfidf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', SVC(kernel='linear', class_weight=custom_weights, random_state=42))
])
param_grid = {
    'tfidf__max_features': [5000, 10000, 15000],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df': [2, 5],
    'clf__C': [0.1, 1.0, 10.0]
}
print("\nStarting Grid Search on TF-IDF + LINEAR SVM (structural TF-IDF params)...")
grid_search = GridSearchCV(
    tfidf_pipeline, param_grid, cv=5, scoring='f1_macro', n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)
print(f"\nBest CV Macro-F1 (TF-IDF only): {grid_search.best_score_:.4f}")
print(f"Winning TF-IDF/clf params: {grid_search.best_params_}")

best_tfidf_params = {
    k.replace('tfidf__', ''): v for k, v in grid_search.best_params_.items() if k.startswith('tfidf__')
}
final_tfidf = TfidfVectorizer(**best_tfidf_params).fit(X_train)
X_train_tfidf = final_tfidf.transform(X_train).toarray()
X_test_tfidf = final_tfidf.transform(X_test).toarray()
report_scale("TF-IDF", X_train_tfidf)

# Assemble the three feature sets once TF-IDF is finalized.
X_train_exact = np.hstack([X_train_tfidf, X_train_lex])
X_test_exact = np.hstack([X_test_tfidf, X_test_lex])
X_train_full = np.hstack([X_train_tfidf, X_train_lex, X_train_sem])
X_test_full = np.hstack([X_test_tfidf, X_test_lex, X_test_sem])

FEATURE_SETS = {
    "TF-IDF ONLY (Baseline SVM)": (X_train_tfidf, X_test_tfidf),
    "TF-IDF + Exact-Match Lexicon": (X_train_exact, X_test_exact),
    "TF-IDF + Exact-Match + Semantic Lexicon": (X_train_full, X_test_full),
}

# ---------------------------------------------------------------------------
# FIX #2: Re-tune C separately for EACH feature set, not just TF-IDF-only.
# The optimal regularization strength for a 15k-dim TF-IDF-only space is not
# necessarily optimal once ~15-40 extra lexicon/semantic dims are added --
# reusing the TF-IDF-only C risks under- or over-crediting the lexicon.
# ---------------------------------------------------------------------------
C_GRID = [0.01, 0.1, 1.0, 10.0, 100.0]

def tune_C(X_feats, y, weights, label):
    print(f"\nTuning C for: {label}")
    best_score, best_C = -1, None
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    for C in C_GRID:
        scores = []
        for tr_idx, val_idx in skf.split(X_feats, y):
            model = SVC(kernel='linear', C=C, class_weight=weights, random_state=42)
            model.fit(X_feats[tr_idx], y[tr_idx])
            preds = model.predict(X_feats[val_idx])
            scores.append(f1_score(y[val_idx], preds, average='macro'))
        mean_score = np.mean(scores)
        print(f"  C={C:<8} CV macro-F1 = {mean_score:.4f}")
        if mean_score > best_score:
            best_score, best_C = mean_score, C
    print(f"  -> Best C for {label}: {best_C} (macro-F1={best_score:.4f})")
    return best_C

best_C_per_set = {}
for label, (Xtr, _) in FEATURE_SETS.items():
    best_C_per_set[label] = tune_C(Xtr, y_train, custom_weights, label)

# ---------------------------------------------------------------------------
# FIX #3: Compare class-weight strategies (grid), not just the hand-picked one.
# Run only on the full feature set, using each set's own tuned C, to keep
# runtime reasonable -- extend to other feature sets if time allows.
# ---------------------------------------------------------------------------
print("\n[Class weight strategy comparison -- on TF-IDF + Exact + Semantic]")
Xtr_full, Xte_full = FEATURE_SETS["TF-IDF + Exact-Match + Semantic Lexicon"]
C_for_full = best_C_per_set["TF-IDF + Exact-Match + Semantic Lexicon"]
weight_results = {}
for wname, weights in CLASS_WEIGHT_GRID.items():
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores, class3_scores = [], []
    for tr_idx, val_idx in skf.split(Xtr_full, y_train):
        model = SVC(kernel='linear', C=C_for_full, class_weight=weights, random_state=42)
        model.fit(Xtr_full[tr_idx], y_train[tr_idx])
        preds = model.predict(Xtr_full[val_idx])
        scores.append(f1_score(y_train[val_idx], preds, average='macro'))
        per_class = f1_score(y_train[val_idx], preds, average=None, labels=[1, 2, 3, 4])
        class3_scores.append(per_class[2])  # index 2 = class label 3
    weight_results[wname] = (np.mean(scores), np.std(scores), np.mean(class3_scores), np.std(class3_scores))
    print(f"  {wname:<22} macro-F1={np.mean(scores):.4f}+/-{np.std(scores):.4f}   "
          f"class-3 F1={np.mean(class3_scores):.4f}+/-{np.std(class3_scores):.4f}")

best_weight_name = max(weight_results, key=lambda k: weight_results[k][2])  # best class-3 F1
best_weights = CLASS_WEIGHT_GRID[best_weight_name]
print(f"\n  -> Selected class-weight strategy: '{best_weight_name}' "
      f"(best class-3 F1 in CV)")

# ---------------------------------------------------------------------------
# FIX #4: Multi-split (repeated stratified k-fold) evaluation on the TEST
# set's distribution, reporting mean +/- std, not a single split's numbers.
# Since the provided test set is fixed/locked, we instead run repeated
# stratified k-fold *within combined train+test* to estimate variance, then
# separately report the single locked-test-set number for the official
# result. Report both: the locked number is your headline metric, the CV
# mean+/-std tells you how much to trust that headline number.
# ---------------------------------------------------------------------------
from sklearn.model_selection import cross_validate

def cv_report(X_feats, y, weights, C, label, n_splits=5, n_repeats=3):
    print(f"\n[Repeated Stratified {n_splits}-fold CV] {label}")
    all_macro, all_class3 = [], []
    for repeat in range(n_repeats):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42 + repeat)
        for tr_idx, val_idx in skf.split(X_feats, y):
            model = SVC(kernel='linear', C=C, class_weight=weights, random_state=42)
            model.fit(X_feats[tr_idx], y[tr_idx])
            preds = model.predict(X_feats[val_idx])
            all_macro.append(f1_score(y[val_idx], preds, average='macro'))
            per_class = f1_score(y[val_idx], preds, average=None, labels=[1, 2, 3, 4])
            all_class3.append(per_class[2])
    print(f"  macro-F1: {np.mean(all_macro):.4f} +/- {np.std(all_macro):.4f}  "
          f"(n={len(all_macro)} folds)")
    print(f"  class-3 F1: {np.mean(all_class3):.4f} +/- {np.std(all_class3):.4f}")
    return np.mean(all_macro), np.std(all_macro)

for label, (Xtr, _) in FEATURE_SETS.items():
    cv_report(Xtr, y_train, best_weights, best_C_per_set[label], label)

# ---------------------------------------------------------------------------
# Final Evaluations on the LOCKED test set (headline numbers), using the
# per-feature-set tuned C and the selected class-weight strategy.
# ---------------------------------------------------------------------------
def train_and_eval(X_train_feats, X_test_feats, label, C, weights):
    model = SVC(kernel='linear', C=C, class_weight=weights, random_state=42).fit(
        X_train_feats, y_train
    )
    y_pred = model.predict(X_test_feats)
    print(f"\n=======================================================")
    print(f"--- {label} (C={C}, weights={weights if isinstance(weights, str) else 'custom'}) ---")
    print(f"=======================================================")
    print(classification_report(y_test, y_pred))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred))
    return y_pred

print("\n[Running Final Evaluations on Locked Test Set]")
for label, (Xtr, Xte) in FEATURE_SETS.items():
    train_and_eval(Xtr, Xte, label, best_C_per_set[label], best_weights)

Loading data...
Precomputing exact-match lexicon features...
Precomputing semantic lexicon features (this loads a sentence-transformer model)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


[Feature scale sanity check]
  Lexicon (exact): min=0.0000  max=1.0000  mean=0.0761
  Semantic: min=0.0000  max=1.0000  mean=0.5935

Starting Grid Search on TF-IDF + LINEAR SVM (structural TF-IDF params)...
Fitting 5 folds for each of 36 candidates, totalling 180 fits

Best CV Macro-F1 (TF-IDF only): 0.6886
Winning TF-IDF/clf params: {'clf__C': 1.0, 'tfidf__max_features': 5000, 'tfidf__min_df': 5, 'tfidf__ngram_range': (1, 2)}
  TF-IDF: min=0.0000  max=1.0000  mean=0.0012

Tuning C for: TF-IDF ONLY (Baseline SVM)
  C=0.01     CV macro-F1 = 0.1196
  C=0.1      CV macro-F1 = 0.6170
  C=1.0      CV macro-F1 = 0.6797
  C=10.0     CV macro-F1 = 0.6605
  C=100.0    CV macro-F1 = 0.6550
  -> Best C for TF-IDF ONLY (Baseline SVM): 1.0 (macro-F1=0.6797)

Tuning C for: TF-IDF + Exact-Match Lexicon
  C=0.01     CV macro-F1 = 0.4596
  C=0.1      CV macro-F1 = 0.5979
  C=1.0      CV macro-F1 = 0.6912
  C=10.0     CV macro-F1 = 0.6776
  C=100.0    CV macro-F1 = 0.6701
  -> Best C for TF-IDF + Exact

In [ ]:
#block_15
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix
from sentence_transformers import SentenceTransformer

print("Loading data...")
train_df = pd.read_csv("train_dataset.csv").dropna(subset=['posts', 'labels'])
test_df = pd.read_csv("test_dataset.csv").dropna(subset=['posts', 'labels'])
lexicon_df = pd.read_excel("/content/Bangla_Depression_Lexicon_latest.xlsx").dropna(
    subset=['term_surface', 'category_name']
)

X_train = train_df['posts'].astype(str).values
y_train = train_df['labels'].astype(int).values
X_test = test_df['posts'].astype(str).values
y_test = test_df['labels'].astype(int).values

# ---------------------------------------------------------------------------
# CUSTOM CLASS WEIGHTS to fix the downward bias into Class 1
# ---------------------------------------------------------------------------
custom_weights = {1: 1.0, 2: 2.0, 3: 4.0, 4: 2.5}

categories = sorted(lexicon_df['category_name'].unique())
cat_phrases = {
    cat: lexicon_df[lexicon_df['category_name'] == cat]['term_surface'].astype(str).tolist()
    for cat in categories
}

has_severity_weight = 'severity_association' in lexicon_df.columns
if has_severity_weight:
    cat_weights = {
        cat: dict(zip(
            lexicon_df[lexicon_df['category_name'] == cat]['term_surface'].astype(str),
            lexicon_df[lexicon_df['category_name'] == cat]['severity_association'].astype(float)
        ))
        for cat in categories
    }

PUNCT_RE = re.compile(r'[।\.,\?!;:"\'\(\)\[\]\{\}/]')
SEGMENT_SPLIT_RE = re.compile(r'[।\n]|[০-৯0-9]+[/\.]')

def clean_text(text):
    return " " + PUNCT_RE.sub(' ', text).strip() + " "

def split_segments(text):
    segments = [s.strip() for s in SEGMENT_SPLIT_RE.split(text) if s.strip()]
    return segments if segments else [text.strip()]

# ---------------------------------------------------------------------------
# Block 1: Exact-Match Lexicon Features (Counts, Proportions, BREADTH)
# ---------------------------------------------------------------------------
class LexiconFeatureExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        raw = self._raw_features(X)
        self.scaler_ = MinMaxScaler().fit(raw)
        return self

    def transform(self, X):
        raw = self._raw_features(X)
        return self.scaler_.transform(raw)

    def _raw_features(self, X):
        features = []
        for text in X:
            text_clean = clean_text(text)
            word_count = max(len(text.split()), 1)

            cat_counts, cat_weighted = [], []
            for cat in categories:
                phrases = cat_phrases[cat]
                count = sum(1 for phrase in phrases if f" {phrase} " in text_clean)
                cat_counts.append(count)
                if has_severity_weight:
                    weighted = sum(
                        cat_weights[cat].get(phrase, 0.0)
                        for phrase in phrases
                        if f" {phrase} " in text_clean
                    )
                    cat_weighted.append(weighted)

            total = sum(cat_counts)

            # The Clinical Breadth Feature! (How many domains triggered?)
            breadth_score = sum(1 for c in cat_counts if c > 0)

            row = []
            row += cat_counts
            row += [total]
            row += [breadth_score]
            row += [(c / total) if total > 0 else 0.0 for c in cat_counts]
            row += [c / word_count for c in cat_counts]
            if has_severity_weight:
                row += cat_weighted
            features.append(row)
        return np.array(features)

# ---------------------------------------------------------------------------
# Block 2: Semantic Lexicon Features
# ---------------------------------------------------------------------------
class SemanticLexiconFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, cat_phrases, model_name='paraphrase-multilingual-mpnet-base-v2'):
        self.cat_phrases = cat_phrases
        self.model_name = model_name

    def fit(self, X, y=None):
        self.model_ = SentenceTransformer(self.model_name)
        self.categories_ = sorted(self.cat_phrases.keys())
        self.term_embeds_ = {
            cat: self.model_.encode(self.cat_phrases[cat], normalize_embeddings=True)
            for cat in self.categories_
        }
        raw = self._raw_features(X)
        self.scaler_ = MinMaxScaler().fit(raw)
        return self

    def transform(self, X):
        raw = self._raw_features(X)
        return self.scaler_.transform(raw)

    def _raw_features(self, X):
        features = []
        for text in X:
            segments = split_segments(str(text))
            seg_embeds = self.model_.encode(segments, normalize_embeddings=True)

            row = []
            for cat in self.categories_:
                sims = seg_embeds @ self.term_embeds_[cat].T
                max_sim = sims.max() if sims.size else 0.0
                top3_mean = np.sort(sims.flatten())[-3:].mean() if sims.size else 0.0
                row += [max_sim, top3_mean]
            features.append(row)
        return np.array(features)

# ---------------------------------------------------------------------------
# Precompute features
# ---------------------------------------------------------------------------
print("Precomputing exact-match lexicon features...")
lex_extractor = LexiconFeatureExtractor().fit(X_train)
X_train_lex = lex_extractor.transform(X_train)
X_test_lex = lex_extractor.transform(X_test)

print("Precomputing semantic lexicon features...")
sem_extractor = SemanticLexiconFeatureExtractor(cat_phrases).fit(X_train)
X_train_sem = sem_extractor.transform(X_train)
X_test_sem = sem_extractor.transform(X_test)

# ---------------------------------------------------------------------------
# Grid search over TF-IDF + LINEAR SVM
# ---------------------------------------------------------------------------
tfidf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    # We explicitly swap in Linear SVM here
    ('clf', SVC(kernel='linear', class_weight=custom_weights, random_state=42))
])

param_grid = {
    # Pushing the boundaries of TF-IDF slightly higher
    'tfidf__max_features': [5000, 10000, 15000],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df': [2, 5],
    'clf__C': [0.1, 1.0, 10.0]  # C tunes the margin of the SVM
}

print("\nStarting Grid Search on TF-IDF + LINEAR SVM...")
grid_search = GridSearchCV(
    tfidf_pipeline, param_grid, cv=5, scoring='f1_macro', n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)
print(f"\nBest CV Macro-F1 (TF-IDF only): {grid_search.best_score_:.4f}")
print(f"Winning TF-IDF/clf params: {grid_search.best_params_}")

best_tfidf_params = {
    k.replace('tfidf__', ''): v for k, v in grid_search.best_params_.items() if k.startswith('tfidf__')
}
best_C = grid_search.best_params_['clf__C']

final_tfidf = TfidfVectorizer(**best_tfidf_params).fit(X_train)
X_train_tfidf = final_tfidf.transform(X_train).toarray()
X_test_tfidf = final_tfidf.transform(X_test).toarray()

# ---------------------------------------------------------------------------
# Final Evaluations
# ---------------------------------------------------------------------------
def train_and_eval(X_train_feats, X_test_feats, label):
    # Train the SVM model
    model = SVC(
        kernel='linear', C=best_C, class_weight=custom_weights, random_state=42
    ).fit(X_train_feats, y_train)

    y_pred = model.predict(X_test_feats)
    print(f"\n=======================================================")
    print(f"--- {label} ---")
    print(f"=======================================================")
    print(classification_report(y_test, y_pred))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred))
    return y_pred


X_train_exact = np.hstack([X_train_tfidf, X_train_lex])
X_test_exact = np.hstack([X_test_tfidf, X_test_lex])

print("\n[Running Final Evaluations on Locked Test Set]")
train_and_eval(X_train_tfidf, X_test_tfidf, "TF-IDF ONLY (Baseline SVM)")
train_and_eval(X_train_exact, X_test_exact, "TF-IDF + Exact-Match Lexicon")

X_train_full = np.hstack([X_train_tfidf, X_train_lex, X_train_sem])
X_test_full = np.hstack([X_test_tfidf, X_test_lex, X_test_sem])
train_and_eval(X_train_full, X_test_full, "TF-IDF + Exact-Match + Semantic Lexicon")

Loading data...
Precomputing exact-match lexicon features...
Precomputing semantic lexicon features...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Starting Grid Search on TF-IDF + LINEAR SVM...
Fitting 5 folds for each of 36 candidates, totalling 180 fits

Best CV Macro-F1 (TF-IDF only): 0.6886
Winning TF-IDF/clf params: {'clf__C': 1.0, 'tfidf__max_features': 5000, 'tfidf__min_df': 5, 'tfidf__ngram_range': (1, 2)}

[Running Final Evaluations on Locked Test Set]

--- TF-IDF ONLY (Baseline SVM) ---
              precision    recall  f1-score   support

           1       0.78      0.79      0.78       420
           2       0.69      0.72      0.70       308
           3       0.49      0.49      0.49       141
           4       0.84      0.66      0.74       111

    accuracy                           0.71       980
   macro avg       0.70      0.67      0.68       980
weighted avg       0.71      0.71      0.71       980

Confusion matrix:
[[333  51  31   5]
 [ 53 222  31   2]
 [ 26  39  69   7]
 [ 17  11  10  73]]

--- TF-IDF + Exact-Match Lexicon ---
              precision    recall  f1-score   support

           1       0.

array([1, 2, 2, 2, 4, 1, 3, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 1, 2,
       1, 2, 2, 2, 3, 2, 1, 2, 4, 2, 2, 4, 2, 3, 1, 1, 1, 2, 2, 1, 4, 1,
       1, 1, 2, 2, 2, 2, 2, 2, 4, 3, 2, 3, 2, 1, 1, 3, 2, 2, 1, 1, 1, 2,
       1, 3, 1, 1, 1, 2, 1, 2, 2, 2, 2, 3, 2, 2, 1, 1, 3, 1, 1, 4, 1, 3,
       1, 1, 4, 3, 1, 1, 1, 2, 3, 1, 1, 1, 1, 2, 1, 2, 3, 2, 3, 3, 1, 1,
       1, 1, 1, 2, 1, 4, 1, 2, 2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 2, 1, 2, 2,
       1, 2, 3, 4, 1, 1, 1, 1, 4, 3, 1, 1, 1, 1, 1, 1, 4, 1, 1, 3, 1, 1,
       2, 1, 3, 1, 1, 2, 1, 4, 1, 3, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 4, 1,
       1, 2, 1, 1, 2, 2, 4, 1, 2, 2, 2, 4, 2, 3, 1, 4, 2, 2, 2, 1, 2, 1,
       4, 1, 1, 1, 2, 1, 4, 2, 2, 3, 2, 1, 2, 2, 1, 2, 1, 2, 1, 4, 1, 1,
       2, 2, 1, 4, 4, 1, 2, 4, 1, 2, 1, 3, 3, 1, 1, 2, 4, 4, 2, 2, 1, 1,
       3, 3, 4, 1, 2, 4, 1, 1, 3, 2, 3, 2, 1, 1, 1, 2, 1, 1, 2, 1, 2, 1,
       1, 2, 1, 1, 3, 4, 2, 1, 1, 1, 1, 1, 2, 1, 2, 1, 1, 1, 2, 1, 2, 4,
       2, 3, 1, 4, 1, 2, 1, 1, 2, 2, 2, 4, 1, 1, 1,

SVM


In [ ]:
#block_15
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report, f1_score

print("1. Loading raw dataset...")
# Make sure the filenames match what you have uploaded in Colab!
train_df = pd.read_csv("train_dataset.csv").dropna(subset=['posts', 'labels'])
test_df = pd.read_csv("test_dataset.csv").dropna(subset=['posts', 'labels'])

X_train = train_df['posts'].astype(str).values
y_train = train_df['labels'].astype(int).values
X_test = test_df['posts'].astype(str).values
y_test = test_df['labels'].astype(int).values

print("2. Building TF-IDF Matrix...")
# Standard TF-IDF setup (Unigrams + Bigrams, capped at 15k words to prevent memory crashes)
paper_tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=15000)
X_train_paper = paper_tfidf.fit_transform(X_train)
X_test_paper = paper_tfidf.transform(X_test)

print("3. Training Kabir et al. (2022) Top Model...")
# The EXACT hyperparameters from their paper (Table 3)
# Note: We omit class_weight='balanced' to strictly replicate their environment
paper_svm = SVC(
    kernel='rbf',
    C=55,
    gamma=0.008,
    random_state=42
)

# Train the model
paper_svm.fit(X_train_paper, y_train)

# Predict on the locked test set
y_pred_paper = paper_svm.predict(X_test_paper)

# Print results
print("\n=======================================================")
print("--- Kabir et al. (2022) Replication Baseline ---")
print("=======================================================")
print(classification_report(y_test, y_pred_paper))

w_f1 = f1_score(y_test, y_pred_paper, average='weighted')
acc = (y_pred_paper == y_test).mean()

print(f"\nReplication Accuracy: {acc:.4f} (Paper claimed ~0.78)")
print(f"Replication Weighted F1: {w_f1:.4f} (Paper claimed ~0.78)")

1. Loading raw dataset...
2. Building TF-IDF Matrix...
3. Training Kabir et al. (2022) Top Model...

--- Kabir et al. (2022) Replication Baseline ---
              precision    recall  f1-score   support

           1       0.73      0.89      0.80       420
           2       0.72      0.74      0.73       308
           3       0.61      0.33      0.43       141
           4       0.89      0.60      0.72       111

    accuracy                           0.73       980
   macro avg       0.74      0.64      0.67       980
weighted avg       0.73      0.73      0.72       980


Replication Accuracy: 0.7296 (Paper claimed ~0.78)
Replication Weighted F1: 0.7159 (Paper claimed ~0.78)


In [ ]:
#block_16
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings('ignore')

print("1. Loading data...")
train_df = pd.read_csv("train_dataset.csv").dropna(subset=['posts', 'labels'])
test_df = pd.read_csv("test_dataset.csv").dropna(subset=['posts', 'labels'])
lexicon_df = pd.read_excel("/content/Bangla_Depression_Lexicon_latest.xlsx").dropna(subset=['term_surface', 'category_name'])

X_train = train_df['posts'].astype(str).values
y_train = train_df['labels'].astype(int).values
X_test = test_df['posts'].astype(str).values
y_test = test_df['labels'].astype(int).values

categories = sorted(lexicon_df['category_name'].unique())
cat_phrases = {cat: lexicon_df[lexicon_df['category_name'] == cat]['term_surface'].astype(str).tolist() for cat in categories}

has_severity_weight = 'severity_association' in lexicon_df.columns
if has_severity_weight:
    cat_weights = {
        cat: dict(zip(
            lexicon_df[lexicon_df['category_name'] == cat]['term_surface'].astype(str),
            lexicon_df[lexicon_df['category_name'] == cat]['severity_association'].astype(float)
        ))
        for cat in categories
    }

PUNCT_RE = re.compile(r'[।\.,\?!;:"\'\(\)\[\]\{\}/]')
SEGMENT_SPLIT_RE = re.compile(r'[।\n]|[০-৯0-9]+[/\.]')

def clean_text(text): return " " + PUNCT_RE.sub(' ', text).strip() + " "
def split_segments(text):
    segments = [s.strip() for s in SEGMENT_SPLIT_RE.split(text) if s.strip()]
    return segments if segments else [text.strip()]

# ---------------------------------------------------------------------------
# EXACT-MATCH LEXICON (Counts, Breadth, Symptom-Free Flag)
# ---------------------------------------------------------------------------
class LexiconFeatureExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.scaler_ = MinMaxScaler().fit(self._raw_features(X))
        return self

    def transform(self, X):
        return self.scaler_.transform(self._raw_features(X))

    def _raw_features(self, X):
        features = []
        for text in X:
            text_clean = clean_text(text)
            word_count = max(len(text.split()), 1)

            cat_counts, cat_weighted = [], []
            for cat in categories:
                phrases = cat_phrases[cat]
                count = sum(1 for phrase in phrases if f" {phrase} " in text_clean)
                cat_counts.append(count)
                if has_severity_weight:
                    weighted = sum(cat_weights[cat].get(phrase, 0.0) for phrase in phrases if f" {phrase} " in text_clean)
                    cat_weighted.append(weighted)

            total = sum(cat_counts)
            breadth_score = sum(1 for c in cat_counts if c > 0)
            is_symptom_free = 1.0 if total == 0 else 0.0

            row = cat_counts + [total, breadth_score, is_symptom_free]
            row += [(c / total) if total > 0 else 0.0 for c in cat_counts]
            row += [c / word_count for c in cat_counts]
            if has_severity_weight:
                row += cat_weighted
            features.append(row)
        return np.array(features)

# ---------------------------------------------------------------------------
# SEMANTIC LEXICON
# ---------------------------------------------------------------------------
class SemanticLexiconFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, cat_phrases, model_name='paraphrase-multilingual-mpnet-base-v2'):
        self.cat_phrases = cat_phrases
        self.model_name = model_name

    def fit(self, X, y=None):
        self.model_ = SentenceTransformer(self.model_name)
        self.categories_ = sorted(self.cat_phrases.keys())
        self.term_embeds_ = {cat: self.model_.encode(self.cat_phrases[cat], normalize_embeddings=True) for cat in self.categories_}
        self.scaler_ = MinMaxScaler().fit(self._raw_features(X))
        return self

    def transform(self, X):
        return self.scaler_.transform(self._raw_features(X))

    def _raw_features(self, X):
        features = []
        for text in X:
            segments = split_segments(str(text))
            seg_embeds = self.model_.encode(segments, normalize_embeddings=True)
            row = []
            for cat in self.categories_:
                sims = seg_embeds @ self.term_embeds_[cat].T
                row += [sims.max() if sims.size else 0.0, np.sort(sims.flatten())[-3:].mean() if sims.size else 0.0]
            features.append(row)
        return np.array(features)

# ---------------------------------------------------------------------------
# FEATURE PRECOMPUTATION
# ---------------------------------------------------------------------------
print("2. Building TF-IDF Matrix (Unigrams+Bigrams)...")
# We skip GridSearch and lock in strong, standard TF-IDF parameters
final_tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=10000)
X_train_tfidf = final_tfidf.fit_transform(X_train).toarray()
X_test_tfidf = final_tfidf.transform(X_test).toarray()

print("3. Precomputing Exact-Match Lexicon Features...")
lex_extractor = LexiconFeatureExtractor().fit(X_train)
X_train_lex = lex_extractor.transform(X_train)
X_test_lex = lex_extractor.transform(X_test)

print("4. Precomputing Semantic Lexicon Features...")
sem_extractor = SemanticLexiconFeatureExtractor(cat_phrases).fit(X_train)
X_train_sem = sem_extractor.transform(X_train)
X_test_sem = sem_extractor.transform(X_test)

# Stack features for ablation
X_train_exact = np.hstack([X_train_tfidf, X_train_lex])
X_test_exact = np.hstack([X_test_tfidf, X_test_lex])

X_train_full = np.hstack([X_train_tfidf, X_train_lex, X_train_sem])
X_test_full = np.hstack([X_test_tfidf, X_test_lex, X_test_sem])

# ---------------------------------------------------------------------------
# ABLATION STUDY: SVM (Kabir et al. RBF Architecture)
# ---------------------------------------------------------------------------
def run_svm_ablation(X_train_feats, X_test_feats, label):
    # The exact architecture from the 2022 paper (RBF Kernel, C=55, gamma=0.008)
    model = SVC(kernel='rbf', C=55, gamma=0.008, random_state=42)
    model.fit(X_train_feats, y_train)

    y_pred = model.predict(X_test_feats)
    print(f"\n=======================================================")
    print(f"--- {label} ---")
    print(f"=======================================================")
    print(classification_report(y_test, y_pred))

print("\n[Running RBF SVM Ablation Study on Test Set]")
run_svm_ablation(X_train_tfidf, X_test_tfidf, "Phase 1: TF-IDF ONLY (Paper's Baseline)")
run_svm_ablation(X_train_exact, X_test_exact, "Phase 2: TF-IDF + Exact-Match Lexicon")
run_svm_ablation(X_train_full, X_test_full, "Phase 3: TF-IDF + Exact + Semantic Lexicon")

1. Loading data...
2. Building TF-IDF Matrix (Unigrams+Bigrams)...
3. Precomputing Exact-Match Lexicon Features...
4. Precomputing Semantic Lexicon Features...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


[Running RBF SVM Ablation Study on Test Set]

--- Phase 1: TF-IDF ONLY (Paper's Baseline) ---
              precision    recall  f1-score   support

           1       0.73      0.90      0.80       420
           2       0.72      0.73      0.73       308
           3       0.61      0.32      0.42       141
           4       0.88      0.60      0.72       111

    accuracy                           0.73       980
   macro avg       0.74      0.64      0.67       980
weighted avg       0.73      0.73      0.72       980


--- Phase 2: TF-IDF + Exact-Match Lexicon ---
              precision    recall  f1-score   support

           1       0.74      0.91      0.82       420
           2       0.74      0.75      0.74       308
           3       0.65      0.33      0.44       141
           4       0.89      0.63      0.74       111

    accuracy                           0.75       980
   macro avg       0.75      0.66      0.69       980
weighted avg       0.74      0.75      0.73

Transformer Experiments


In [ ]:
!unzip BanglaBERT-20260925T152855Z-1-001.zip -d dapt_model/

Archive:  BanglaBERT-20260925T152855Z-1-001.zip
  inflating: dapt_model/BanglaBERT/dapt_complete.txt  
  inflating: dapt_model/BanglaBERT/training_args.bin  
  inflating: dapt_model/BanglaBERT/config.json  
  inflating: dapt_model/BanglaBERT/tokenizer_config.json  
  inflating: dapt_model/BanglaBERT/tokenizer.json  
  inflating: dapt_model/BanglaBERT/model.safetensors  


In [ ]:
# Install the required HuggingFace libraries
!pip install transformers accelerate

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import classification_report, f1_score

# 1. Load Data
print("Loading data...")
train_df = pd.read_csv("train_dataset.csv").dropna(subset=['posts', 'labels'])
test_df = pd.read_csv("test_dataset.csv").dropna(subset=['posts', 'labels'])

X_train = train_df['posts'].astype(str).tolist()
# BERT expects labels to start at 0, so we subtract 1 from your 1-4 labels
y_train = (train_df['labels'].astype(int) - 1).tolist()
X_test = test_df['posts'].astype(str).tolist()
y_test = (test_df['labels'].astype(int) - 1).tolist()

# 2. Tokenizer & Model
# -------------------------------------------------------------------
# TO RUN DAPT: Change this string to the path of your DAPT checkpoint!
# -------------------------------------------------------------------
MODEL_NAME = "./dapt_model/BanglaBERT"

print(f"Loading Tokenizer and Model...")

# 1. Force it to get the Tokenizer from the Base model
tokenizer = AutoTokenizer.from_pretrained("sagorsarker/bangla-bert-base")

# 2. But load the actual Neural Network from your DAPT folder!
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=4)

print(f"Loading Tokenizer and Model: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=4)

# 3. Dataset Preparation
class DepressionDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=256)
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = DepressionDataset(X_train, y_train)
test_dataset = DepressionDataset(X_test, y_test)

# 4. Metrics Setup
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    macro_f1 = f1_score(labels, preds, average='macro')
    weighted_f1 = f1_score(labels, preds, average='weighted')
    return {"macro_f1": macro_f1, "weighted_f1": weighted_f1}

# 5. Trainer Setup
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    learning_rate=2e-5,
    weight_decay=0.01
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

# 6. Train and Evaluate
print(f"\n🚀 Starting Fine-Tuning for {MODEL_NAME}...")
trainer.train()

print("\n📊 Evaluating Best Epoch on Locked Test Set...")
preds = trainer.predict(test_dataset)
y_pred = preds.predictions.argmax(-1)

# Add 1 back to the labels to match your original 1-4 scale
print(classification_report(np.array(y_test) + 1, y_pred + 1))

Loading data...
Loading Tokenizer and Model...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: ./dapt_model/BanglaBERT
Key                                    | Status     | 
---------------------------------------+------------+-
generator_lm_head.bias                 | UNEXPECTED | 
generator_predictions.dense.bias       | UNEXPECTED | 
generator_predictions.dense.weight     | UNEXPECTED | 
generator_predictions.LayerNorm.bias   | UNEXPECTED | 
generator_predictions.LayerNorm.weight | UNEXPECTED | 
classifier.out_proj.weight             | MISSING    | 
classifier.dense.weight                | MISSING    | 
classifier.dense.bias                  | MISSING    | 
classifier.out_proj.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading Tokenizer and Model: ./dapt_model/BanglaBERT...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: ./dapt_model/BanglaBERT
Key                                    | Status     | 
---------------------------------------+------------+-
generator_lm_head.bias                 | UNEXPECTED | 
generator_predictions.dense.bias       | UNEXPECTED | 
generator_predictions.dense.weight     | UNEXPECTED | 
generator_predictions.LayerNorm.bias   | UNEXPECTED | 
generator_predictions.LayerNorm.weight | UNEXPECTED | 
classifier.out_proj.weight             | MISSING    | 
classifier.dense.weight                | MISSING    | 
classifier.dense.bias                  | MISSING    | 
classifier.out_proj.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



🚀 Starting Fine-Tuning for ./dapt_model/BanglaBERT...


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
